<a href="https://colab.research.google.com/github/worasak2530seed/fast-hybrid-pro/blob/main/Fast_Hybrid_Pro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================================
# FAST HYBRID PRO v2
# CLEAN RESEARCH / BACKTEST NOTEBOOK
# ============================================================

!pip -q install yfinance pandas numpy matplotlib

import warnings
warnings.filterwarnings("ignore")

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from dataclasses import dataclass, asdict

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 100)

print("✅ Setup complete")

✅ Setup complete


In [ ]:

# ============================================================
# 2) CONFIGURATION
# ============================================================

CAPITAL = 100_000

# Maximum risk per trade
MAX_RISK_PCT = 0.02

# Thai board lot
BOARD_LOT = 100

# Minimum score for signal
MIN_SCORE = 3

# Historical data requirement
MIN_BARS = 200

# Backtest warm-up
WARMUP_BARS = 200

# Maximum holding period
MAX_HOLDING_DAYS = 20

# ============================================================
# PRODUCTION STRATEGY — selected from unseen-test validation
# ============================================================
PRODUCTION_MIN_SCORE = 4
PRODUCTION_WEEKLY_CONFIRM = False
PRODUCTION_ATR_MULT = 1.2
PRODUCTION_TARGET2_PCT = 0.15
PRODUCTION_MAX_HOLDING_DAYS = 20

PRODUCTION_SYMBOLS = [
    "TKN.BK",
    "SAPPE.BK",
    "SPA.BK",
    "XO.BK",
    "FORTH.BK",
    "DITTO.BK",
    "ICHI.BK",
    "MASTER.BK",
]


WATCHLIST = sorted(set([
    'DELTA.BK',
    'SNNP.BK',
    'JMART.BK',
    'JMT.BK',
    'FORTH.BK',
    'SPA.BK',
    'AU.BK',
    'SAPPE.BK',
    'ICHI.BK',
    'CBG.BK',
    'KAMART.BK',
    'MOSHI.BK',
    'MEB.BK',
    'MASTER.BK',
    'KLINIQ.BK',
    'TKN.BK',
    'XO.BK',
    'DITTO.BK',
    'PLANB.BK'
]))


@dataclass
class StrategyConfig:

    # --------------------------------------------------------
    # MACD
    # --------------------------------------------------------
    macd_fast: int = 4
    macd_slow: int = 12
    macd_signal: int = 9

    # --------------------------------------------------------
    # STOCHASTIC
    # --------------------------------------------------------
    stoch_k: int = 8
    stoch_smooth: int = 3
    stoch_d: int = 3

    # --------------------------------------------------------
    # DAILY TREND
    # --------------------------------------------------------
    ema_fast: int = 5
    ema_mid: int = 10
    ema_long: int = 75

    # --------------------------------------------------------
    # WEEKLY TREND
    # --------------------------------------------------------
    weekly_ema_fast: int = 5
    weekly_ema_slow: int = 10

    # --------------------------------------------------------
    # ATR
    # --------------------------------------------------------
    atr_length: int = 14
    atr_stop_mult: float = 1.5

    # --------------------------------------------------------
    # BOLLINGER
    # --------------------------------------------------------
    bb_length: int = 20
    bb_mult: float = 2.0

    # --------------------------------------------------------
    # KELTNER
    # --------------------------------------------------------
    kc_mult: float = 1.5

    # --------------------------------------------------------
    # SQUEEZE
    # --------------------------------------------------------
    squeeze_lookback: int = 10

    # --------------------------------------------------------
    # VOLUME
    # --------------------------------------------------------
    volume_ma: int = 5
    volume_ratio_min: float = 1.2

    # --------------------------------------------------------
    # TARGET
    # --------------------------------------------------------
    target1_pct: float = 0.07
    target2_pct: float = 0.15

    # --------------------------------------------------------
    # SCORE
    # --------------------------------------------------------
    min_score: int = 3


# Current canonical research setup
CANONICAL = StrategyConfig(
    macd_fast=4,
    macd_slow=12,
    macd_signal=9
)


# Alternative setup for comparison
FAST_MACD = StrategyConfig(
    macd_fast=6,
    macd_slow=13,
    macd_signal=5
)


print("WATCHLIST:", len(WATCHLIST))

print("\nCANONICAL")
print(asdict(CANONICAL))

print("\nFAST MACD")
print(asdict(FAST_MACD))

WATCHLIST: 19

CANONICAL
{'macd_fast': 4, 'macd_slow': 12, 'macd_signal': 9, 'stoch_k': 8, 'stoch_smooth': 3, 'stoch_d': 3, 'ema_fast': 5, 'ema_mid': 10, 'ema_long': 75, 'weekly_ema_fast': 5, 'weekly_ema_slow': 10, 'atr_length': 14, 'atr_stop_mult': 1.5, 'bb_length': 20, 'bb_mult': 2.0, 'kc_mult': 1.5, 'squeeze_lookback': 10, 'volume_ma': 5, 'volume_ratio_min': 1.2, 'target1_pct': 0.07, 'target2_pct': 0.15, 'min_score': 3}

FAST MACD
{'macd_fast': 6, 'macd_slow': 13, 'macd_signal': 5, 'stoch_k': 8, 'stoch_smooth': 3, 'stoch_d': 3, 'ema_fast': 5, 'ema_mid': 10, 'ema_long': 75, 'weekly_ema_fast': 5, 'weekly_ema_slow': 10, 'atr_length': 14, 'atr_stop_mult': 1.5, 'bb_length': 20, 'bb_mult': 2.0, 'kc_mult': 1.5, 'squeeze_lookback': 10, 'volume_ma': 5, 'volume_ratio_min': 1.2, 'target1_pct': 0.07, 'target2_pct': 0.15, 'min_score': 3}


In [ ]:

# ============================================================
# 3) DATA ENGINE
# ============================================================

REQUIRED_COLS = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume"
]


def normalize_ohlcv(df):

    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()

    # yfinance can return MultiIndex columns
    if isinstance(out.columns, pd.MultiIndex):
        out.columns = out.columns.get_level_values(0)

    # Remove duplicate columns
    out = out.loc[:, ~out.columns.duplicated()]

    missing = [
        c for c in REQUIRED_COLS
        if c not in out.columns
    ]

    if missing:
        raise ValueError(
            f"Missing OHLCV columns: {missing}"
        )

    out = out[REQUIRED_COLS].copy()

    # Datetime
    out.index = pd.to_datetime(out.index)

    # Remove timezone if present
    if getattr(out.index, "tz", None) is not None:
        out.index = out.index.tz_localize(None)

    # Numeric conversion
    for col in REQUIRED_COLS:
        out[col] = pd.to_numeric(
            out[col],
            errors="coerce"
        )

    # Remove invalid rows
    out = out.dropna(
        subset=[
            "Open",
            "High",
            "Low",
            "Close"
        ]
    )

    # Remove duplicate dates
    out = out[
        ~out.index.duplicated(
            keep="last"
        )
    ]

    # Sort chronologically
    out = out.sort_index()

    return out


def download_stock(
    symbol,
    period="5y"
):

    raw = yf.download(
        symbol,
        period=period,
        interval="1d",
        auto_adjust=False,
        progress=False,
        threads=False
    )

    return normalize_ohlcv(raw)


data = {}
errors = {}


print("🚀 Downloading market data...\n")


for symbol in WATCHLIST:

    try:

        df = download_stock(symbol)

        if len(df) < MIN_BARS:

            errors[symbol] = (
                f"Only {len(df)} bars "
                f"(minimum {MIN_BARS})"
            )

        else:

            data[symbol] = df

            print(
                f"✅ {symbol:<12} "
                f"{len(df):>5} bars"
            )

    except Exception as e:

        errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


print("\n====================================")
print(
    f"Loaded: {len(data)} / "
    f"{len(WATCHLIST)}"
)
print("====================================")


if errors:

    print("\n⚠️ Errors / insufficient data:")

    for symbol, error in errors.items():

        print(
            f"{symbol}: {error}"
        )

🚀 Downloading market data...



✅ AU.BK         1219 bars


✅ CBG.BK        1219 bars


✅ DELTA.BK      1219 bars


✅ DITTO.BK      1219 bars


...

✅ SAPPE.BK      1219 bars


✅ SNNP.BK       1219 bars


✅ SPA.BK        1219 bars


✅ TKN.BK        1219 bars


✅ XO.BK         1219 bars

Loaded: 19 / 19


In [ ]:

# ============================================================
# 4) DAILY INDICATORS
# ============================================================

def wilder_rma(series, length):
    """
    Wilder RMA
    ใช้แนวคิดเดียวกับ TradingView ta.rma()
    """

    return series.ewm(
        alpha=1 / length,
        adjust=False,
        min_periods=length
    ).mean()


def calculate_atr(df, length=14):

    prev_close = df["Close"].shift(1)

    tr1 = df["High"] - df["Low"]
    tr2 = (df["High"] - prev_close).abs()
    tr3 = (df["Low"] - prev_close).abs()

    true_range = pd.concat(
        [tr1, tr2, tr3],
        axis=1
    ).max(axis=1)

    atr = wilder_rma(
        true_range,
        length
    )

    return true_range, atr


def calculate_daily_indicators(df):

    out = df.copy()

    close = out["Close"]
    high = out["High"]
    low = out["Low"]
    volume = out["Volume"]

    # --------------------------------------------------------
    # EMA 5 / 10 / 75
    # --------------------------------------------------------

    out["EMA5"] = close.ewm(
        span=5,
        adjust=False
    ).mean()

    out["EMA10"] = close.ewm(
        span=10,
        adjust=False
    ).mean()

    out["EMA75"] = close.ewm(
        span=75,
        adjust=False
    ).mean()

    # --------------------------------------------------------
    # MACD 4 / 12 / 9
    # --------------------------------------------------------

    ema_fast = close.ewm(
        span=4,
        adjust=False
    ).mean()

    ema_slow = close.ewm(
        span=12,
        adjust=False
    ).mean()

    out["MACD"] = ema_fast - ema_slow

    out["MACD_SIGNAL"] = out["MACD"].ewm(
        span=9,
        adjust=False
    ).mean()

    out["MACD_HIST"] = (
        out["MACD"] - out["MACD_SIGNAL"]
    )

    # --------------------------------------------------------
    # STOCHASTIC 8 / 3 / 3
    # --------------------------------------------------------

    lowest_8 = low.rolling(
        window=8,
        min_periods=8
    ).min()

    highest_8 = high.rolling(
        window=8,
        min_periods=8
    ).max()

    price_range = highest_8 - lowest_8

    stoch_raw = (
        100
        * (close - lowest_8)
        / price_range.replace(0, np.nan)
    )

    out["STOCH_K"] = stoch_raw.rolling(
        window=3,
        min_periods=3
    ).mean()

    out["STOCH_D"] = out["STOCH_K"].rolling(
        window=3,
        min_periods=3
    ).mean()

    # --------------------------------------------------------
    # ATR 14 — WILDER
    # --------------------------------------------------------

    out["TR"], out["ATR14"] = calculate_atr(
        out,
        length=14
    )

    # --------------------------------------------------------
    # BOLLINGER BAND 20 / 2
    # --------------------------------------------------------

    bb_mid = close.rolling(
        window=20,
        min_periods=20
    ).mean()

    bb_std = close.rolling(
        window=20,
        min_periods=20
    ).std()

    out["BB_MID"] = bb_mid

    out["BB_UPPER"] = (
        bb_mid + (2.0 * bb_std)
    )

    out["BB_LOWER"] = (
        bb_mid - (2.0 * bb_std)
    )

    # --------------------------------------------------------
    # KELTNER CHANNEL
    # --------------------------------------------------------

    out["KC_MID"] = bb_mid

    out["KC_UPPER"] = (
        bb_mid + (1.5 * out["ATR14"])
    )

    out["KC_LOWER"] = (
        bb_mid - (1.5 * out["ATR14"])
    )

    # --------------------------------------------------------
    # SQUEEZE
    # --------------------------------------------------------

    out["SQUEEZE"] = (
        (out["BB_UPPER"] <= out["KC_UPPER"])
        &
        (out["BB_LOWER"] >= out["KC_LOWER"])
    )

    # --------------------------------------------------------
    # SQUEEZE RELEASE
    # --------------------------------------------------------

    out["SQUEEZE_RELEASE"] = (
        (~out["SQUEEZE"])
        &
        (out["SQUEEZE"].shift(1).fillna(False))
    )

    # --------------------------------------------------------
    # BARS SINCE LAST SQUEEZE
    # --------------------------------------------------------

    bars_since_squeeze = []

    last_squeeze_index = None

    for i, is_squeeze in enumerate(
        out["SQUEEZE"].fillna(False)
    ):

        if is_squeeze:

            last_squeeze_index = i
            bars_since_squeeze.append(0)

        elif last_squeeze_index is None:

            bars_since_squeeze.append(np.nan)

        else:

            bars_since_squeeze.append(
                i - last_squeeze_index
            )

    out["BARS_SINCE_SQUEEZE"] = (
        bars_since_squeeze
    )

    out["SQUEEZE_RECENT"] = (
        out["BARS_SINCE_SQUEEZE"] <= 10
    )

    # --------------------------------------------------------
    # VOLUME
    # --------------------------------------------------------

    out["VOLUME_MA5"] = volume.rolling(
        window=5,
        min_periods=5
    ).mean()

    out["VOLUME_RATIO"] = (
        volume
        / out["VOLUME_MA5"].replace(0, np.nan)
    )

    # --------------------------------------------------------
    # CANDLE BODY
    # --------------------------------------------------------

    out["BODY"] = (
        close - out["Open"]
    ).abs()

    out["BODY_ATR_RATIO"] = (
        out["BODY"]
        / out["ATR14"].replace(0, np.nan)
    )

    return out


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

daily_data = {}
indicator_errors = {}

print("📊 Calculating daily indicators...\n")

for symbol, df in data.items():

    try:

        result = calculate_daily_indicators(df)

        daily_data[symbol] = result

        print(
            f"✅ {symbol:<12} "
            f"{len(result):>5} bars"
        )

    except Exception as e:

        indicator_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


print("\n====================================")
print(
    f"Daily indicator data: "
    f"{len(daily_data)} / {len(data)}"
)
print("====================================")


if indicator_errors:

    print("\n⚠️ Indicator errors:")

    for symbol, error in indicator_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# QUICK VALIDATION
# ============================================================

if daily_data:

    sample_symbol = next(iter(daily_data))

    sample = daily_data[sample_symbol]

    print(
        f"\n🔎 Quick validation: {sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "EMA5",
                "EMA10",
                "EMA75",
                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",
                "STOCH_K",
                "STOCH_D",
                "ATR14",
                "SQUEEZE",
                "SQUEEZE_RELEASE",
                "VOLUME_RATIO"
            ]
        ].tail(5)
    )

📊 Calculating daily indicators...

✅ AU.BK         1219 bars
✅ CBG.BK        1219 bars
✅ DELTA.BK      1219 bars
✅ DITTO.BK      1219 bars
✅ FORTH.BK      1219 bars
✅ ICHI.BK       1219 bars
✅ JMART.BK      1219 bars


✅ JMT.BK        1219 bars
✅ KAMART.BK     1219 bars
✅ KLINIQ.BK      959 bars
✅ MASTER.BK      905 bars
✅ MEB.BK         891 bars
✅ MOSHI.BK       928 bars
✅ PLANB.BK      1219 bars


✅ SAPPE.BK      1219 bars
✅ SNNP.BK       1219 bars
✅ SPA.BK        1219 bars
✅ TKN.BK        1219 bars
✅ XO.BK         1219 bars

Daily indicator data: 19 / 19

🔎 Quick validation: AU.BK


Price,Close,EMA5,EMA10,EMA75,MACD,MACD_SIGNAL,MACD_HIST,STOCH_K,STOCH_D,ATR14,SQUEEZE,SQUEEZE_RELEASE,VOLUME_RATIO
Date,,,,,,,,,,,,,
2026-10-01,4.14,4.175048,4.198474,4.438591,-0.037745,-0.027935,-0.009810,21.666578,24.999886,0.047991,False,True,1.063792
2026-10-02,4.14,4.163366,4.187843,4.430733,-0.038964,-0.030141,-0.008823,16.666617,21.666578,0.051706,False,False,1.187989
2026-10-05,4.20,4.175577,4.190053,4.424662,-0.022416,-0.028596,0.006180,40.476141,26.269779,0.055155,True,False,1.358750
2026-10-06,4.18,4.177051,4.188225,4.418223,-0.017559,-0.026389,0.008830,51.190451,36.111070,0.054073,False,True,1.037210
2026-10-07,4.16,4.171367,4.183093,4.411428,-0.018935,-0.024898,0.005963,57.142857,49.603150,0.054496,False,False,0.469267


In [ ]:

# ============================================================
# 5) WEEKLY INDICATORS
# ============================================================

def calculate_weekly_indicators(df):

    daily = df.copy()

    # --------------------------------------------------------
    # Daily -> Weekly
    #
    # Week ends on Friday.
    # --------------------------------------------------------

    weekly = daily.resample("W-FRI").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "Volume": "sum"
    })

    weekly = weekly.dropna(
        subset=[
            "Open",
            "High",
            "Low",
            "Close"
        ]
    )

    close = weekly["Close"]

    # --------------------------------------------------------
    # WEEKLY EMA 5 / 10
    # --------------------------------------------------------

    weekly["W_EMA5"] = close.ewm(
        span=5,
        adjust=False
    ).mean()

    weekly["W_EMA10"] = close.ewm(
        span=10,
        adjust=False
    ).mean()

    # --------------------------------------------------------
    # WEEKLY MACD 4 / 12 / 9
    # --------------------------------------------------------

    weekly_ema_fast = close.ewm(
        span=4,
        adjust=False
    ).mean()

    weekly_ema_slow = close.ewm(
        span=12,
        adjust=False
    ).mean()

    weekly["W_MACD"] = (
        weekly_ema_fast
        - weekly_ema_slow
    )

    weekly["W_MACD_SIGNAL"] = (
        weekly["W_MACD"].ewm(
            span=9,
            adjust=False
        ).mean()
    )

    weekly["W_MACD_HIST"] = (
        weekly["W_MACD"]
        - weekly["W_MACD_SIGNAL"]
    )

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # A weekly candle ending Friday becomes available
    # only from the next trading day.
    #
    # Example:
    #
    # Week ending 2026-09-25
    # -> usable from 2026-09-28
    #
    # This prevents Friday from seeing its own completed
    # weekly candle during the Friday signal.
    # --------------------------------------------------------

    weekly_features = weekly[
        [
            "W_EMA5",
            "W_EMA10",
            "W_MACD",
            "W_MACD_SIGNAL",
            "W_MACD_HIST"
        ]
    ].copy()

    # Move availability date to next Monday.
    #
    # W-FRI + 3 days = Monday
    #
    # If Monday is a market holiday, forward-fill will keep
    # the last completed weekly value until the next trading day.
    weekly_features.index = (
        weekly_features.index
        + pd.Timedelta(days=3)
    )

    # --------------------------------------------------------
    # Map completed weekly values to daily data
    # --------------------------------------------------------

    daily_index = daily.index

    mapped_weekly = weekly_features.reindex(
        daily_index,
        method="ffill"
    )

    # --------------------------------------------------------
    # Combine
    # --------------------------------------------------------

    out = daily.copy()

    for col in mapped_weekly.columns:
        out[col] = mapped_weekly[col]

    return out, weekly


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

weekly_data = {}
combined_data = {}
weekly_errors = {}

print("📅 Calculating weekly indicators...\n")

for symbol, df in daily_data.items():

    try:

        combined, weekly = calculate_weekly_indicators(df)

        combined_data[symbol] = combined
        weekly_data[symbol] = weekly

        print(
            f"✅ {symbol:<12} "
            f"Daily: {len(combined):>5} | "
            f"Weekly: {len(weekly):>4}"
        )

    except Exception as e:

        weekly_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


print("\n====================================")
print(
    f"Weekly indicator data: "
    f"{len(combined_data)} / {len(daily_data)}"
)
print("====================================")


if weekly_errors:

    print("\n⚠️ Weekly indicator errors:")

    for symbol, error in weekly_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# VALIDATION
# ============================================================

if combined_data:

    sample_symbol = next(iter(combined_data))

    sample = combined_data[sample_symbol]

    print(
        f"\n🔎 Weekly validation: {sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "W_EMA5",
                "W_EMA10",
                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST"
            ]
        ].tail(15)
    )

📅 Calculating weekly indicators...



✅ AU.BK        Daily:  1219 | Weekly:  262
✅ CBG.BK       Daily:  1219 | Weekly:  262
✅ DELTA.BK     Daily:  1219 | Weekly:  262
✅ DITTO.BK     Daily:  1219 | Weekly:  262
✅ FORTH.BK     Daily:  1219 | Weekly:  262
✅ ICHI.BK      Daily:  1219 | Weekly:  262
✅ JMART.BK     Daily:  1219 | Weekly:  262
✅ JMT.BK       Daily:  1219 | Weekly:  262
✅ KAMART.BK    Daily:  1219 | Weekly:  262
✅ KLINIQ.BK    Daily:   959 | Weekly:  205
✅ MASTER.BK    Daily:   905 | Weekly:  194
✅ MEB.BK       Daily:   891 | Weekly:  191
✅ MOSHI.BK     Daily:   928 | Weekly:  199


✅ PLANB.BK     Daily:  1219 | Weekly:  262
✅ SAPPE.BK     Daily:  1219 | Weekly:  262


✅ SNNP.BK      Daily:  1219 | Weekly:  262
✅ SPA.BK       Daily:  1219 | Weekly:  262
✅ TKN.BK       Daily:  1219 | Weekly:  262
✅ XO.BK        Daily:  1219 | Weekly:  262

Weekly indicator data: 19 / 19

🔎 Weekly validation: AU.BK


Price,Close,W_EMA5,W_EMA10,W_MACD,W_MACD_SIGNAL,W_MACD_HIST
Date,,,,,,
2026-09-17,4.24,4.368172,4.466597,-0.152274,-0.090567,-0.061708
2026-09-18,4.24,4.368172,4.466597,-0.152274,-0.090567,-0.061708
2026-09-21,4.24,4.325448,4.425397,-0.153826,-0.103219,-0.050607
2026-09-22,4.24,4.325448,4.425397,-0.153826,-0.103219,-0.050607
2026-09-23,4.22,4.325448,4.425397,-0.153826,-0.103219,-0.050607
2026-09-24,4.22,4.325448,4.425397,-0.153826,-0.103219,-0.050607
2026-09-25,4.20,4.325448,4.425397,-0.153826,-0.103219,-0.050607
2026-09-28,4.20,4.283632,4.384416,-0.154994,-0.113574,-0.041420
2026-09-29,4.20,4.283632,4.384416,-0.154994,-0.113574,-0.041420


In [ ]:

# ============================================================
# 6) BUILD FEATURE DATASETS
# ============================================================

feature_data = {}

feature_errors = {}


for symbol, df in combined_data.items():

    try:

        out = df.copy()

        # ----------------------------------------------------
        # Basic validity
        # ----------------------------------------------------

        required = [
            "Open",
            "High",
            "Low",
            "Close",
            "Volume",

            "EMA5",
            "EMA10",
            "EMA75",

            "MACD",
            "MACD_SIGNAL",
            "MACD_HIST",

            "STOCH_K",
            "STOCH_D",

            "ATR14",

            "BB_UPPER",
            "BB_LOWER",

            "KC_UPPER",
            "KC_LOWER",

            "SQUEEZE",
            "SQUEEZE_RELEASE",
            "SQUEEZE_RECENT",

            "VOLUME_RATIO",

            "W_EMA5",
            "W_EMA10",

            "W_MACD",
            "W_MACD_SIGNAL",
            "W_MACD_HIST"
        ]

        missing = [
            col
            for col in required
            if col not in out.columns
        ]

        if missing:

            raise ValueError(
                f"Missing columns: {missing}"
            )

        # ----------------------------------------------------
        # Remove rows that cannot produce indicators
        #
        # IMPORTANT:
        # We do NOT drop rows globally before calculating
        # indicators. All indicators have already been
        # calculated using the full historical dataset.
        # ----------------------------------------------------

        out = out.dropna(
            subset=[
                "EMA75",
                "ATR14",
                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",
                "STOCH_K",
                "STOCH_D",
                "W_EMA5",
                "W_EMA10",
                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST",
                "VOLUME_RATIO"
            ]
        )

        # ----------------------------------------------------
        # Ensure chronological order
        # ----------------------------------------------------

        out = out.sort_index()

        # ----------------------------------------------------
        # Ensure unique dates
        # ----------------------------------------------------

        out = out[
            ~out.index.duplicated(
                keep="last"
            )
        ]

        # ----------------------------------------------------
        # Create previous-bar values
        #
        # Used later for signal confirmation.
        # ----------------------------------------------------

        out["PREV_CLOSE"] = (
            out["Close"].shift(1)
        )

        out["PREV_MACD"] = (
            out["MACD"].shift(1)
        )

        out["PREV_MACD_SIGNAL"] = (
            out["MACD_SIGNAL"].shift(1)
        )

        out["PREV_MACD_HIST"] = (
            out["MACD_HIST"].shift(1)
        )

        out["PREV_STOCH_K"] = (
            out["STOCH_K"].shift(1)
        )

        out["PREV_STOCH_D"] = (
            out["STOCH_D"].shift(1)
        )

        # ----------------------------------------------------
        # Daily trend state
        # ----------------------------------------------------

        out["TREND_BULL"] = (
            (out["Close"] > out["EMA5"])
            &
            (out["EMA5"] > out["EMA10"])
            &
            (out["EMA10"] > out["EMA75"])
        )

        # ----------------------------------------------------
        # Weekly trend state
        # ----------------------------------------------------

        out["WEEKLY_TREND_BULL"] = (
            out["W_EMA5"] > out["W_EMA10"]
        )

        # ----------------------------------------------------
        # Daily MACD bullish state
        # ----------------------------------------------------

        out["MACD_BULL"] = (
            out["MACD"] > out["MACD_SIGNAL"]
        )

        out["MACD_HIST_RISING"] = (
            out["MACD_HIST"]
            > out["PREV_MACD_HIST"]
        )

        # ----------------------------------------------------
        # Weekly MACD bullish state
        # ----------------------------------------------------

        out["WEEKLY_MACD_BULL"] = (
            out["W_MACD"] > out["W_MACD_SIGNAL"]
        )

        out["WEEKLY_MACD_HIST_RISING"] = (
            out["W_MACD_HIST"]
            > out["W_MACD_HIST"].shift(1)
        )

        # ----------------------------------------------------
        # Stochastic bullish state
        # ----------------------------------------------------

        out["STOCH_BULL"] = (
            (out["STOCH_K"] > out["STOCH_D"])
            &
            (out["STOCH_K"] < 80)
        )

        # ----------------------------------------------------
        # Volume confirmation
        # ----------------------------------------------------

        out["VOLUME_CONFIRM"] = (
            out["VOLUME_RATIO"]
            >= 1.2
        )

        # ----------------------------------------------------
        # Early-entry / squeeze state
        # ----------------------------------------------------

        out["EARLY_ENTRY_ZONE"] = (
            out["SQUEEZE"]
            |
            out["SQUEEZE_RELEASE"]
            |
            out["SQUEEZE_RECENT"]
        )

        # ----------------------------------------------------
        # Target prices
        # ----------------------------------------------------

        out["TARGET1"] = (
            out["Close"] * 1.07
        )

        out["TARGET2"] = (
            out["Close"] * 1.15
        )

        # ----------------------------------------------------
        # Final dataset
        # ----------------------------------------------------

        feature_data[symbol] = out

    except Exception as e:

        feature_errors[symbol] = str(e)


print("📦 Building feature datasets...\n")

for symbol, df in feature_data.items():

    print(
        f"✅ {symbol:<12} "
        f"{len(df):>5} usable bars"
    )


print("\n====================================")
print(
    f"Feature datasets: "
    f"{len(feature_data)} / {len(combined_data)}"
)
print("====================================")


if feature_errors:

    print("\n⚠️ Feature errors:")

    for symbol, error in feature_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# VALIDATION
# ============================================================

if feature_data:

    sample_symbol = next(iter(feature_data))

    sample = feature_data[sample_symbol]

    print(
        f"\n🔎 Feature validation: {sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "EMA5",
                "EMA10",
                "EMA75",
                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",
                "W_EMA5",
                "W_EMA10",
                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST",
                "TREND_BULL",
                "WEEKLY_TREND_BULL",
                "MACD_BULL",
                "STOCH_BULL",
                "VOLUME_CONFIRM",
                "EARLY_ENTRY_ZONE"
            ]
        ].tail(10)
    )

📦 Building feature datasets...

✅ AU.BK         1206 usable bars
✅ CBG.BK        1206 usable bars
✅ DELTA.BK      1206 usable bars
✅ DITTO.BK      1206 usable bars
✅ FORTH.BK      1206 usable bars
✅ ICHI.BK       1206 usable bars
✅ JMART.BK      1206 usable bars
✅ JMT.BK        1206 usable bars
✅ KAMART.BK     1205 usable bars
✅ KLINIQ.BK      946 usable bars
✅ MASTER.BK      892 usable bars
✅ MEB.BK         878 usable bars
✅ MOSHI.BK       915 usable bars
✅ PLANB.BK      1206 usable bars
✅ SAPPE.BK      1206 usable bars
✅ SNNP.BK       1206 usable bars
✅ SPA.BK        1206 usable bars
✅ TKN.BK        1206 usable bars
✅ XO.BK         1206 usable bars

Feature datasets: 19 / 19

🔎 Feature validation: AU.BK


Price,Close,EMA5,EMA10,EMA75,MACD,MACD_SIGNAL,MACD_HIST,W_EMA5,W_EMA10,W_MACD,W_MACD_SIGNAL,W_MACD_HIST,TREND_BULL,WEEKLY_TREND_BULL,MACD_BULL,STOCH_BULL,VOLUME_CONFIRM,EARLY_ENTRY_ZONE
Date,,,,,,,,,,,,,,,,,,
2026-09-24,4.22,4.229900,4.241822,4.475599,-0.020207,-0.024648,0.004441,4.325448,4.425397,-0.153826,-0.103219,-0.050607,False,False,True,False,True,False
2026-09-25,4.20,4.219933,4.234218,4.468346,-0.023887,-0.024496,0.000609,4.325448,4.425397,-0.153826,-0.103219,-0.050607,False,False,True,False,False,False
2026-09-28,4.20,4.213289,4.227997,4.461285,-0.024285,-0.024454,0.000169,4.283632,4.384416,-0.154994,-0.113574,-0.041420,False,False,True,True,True,True
2026-09-29,4.20,4.208859,4.222906,4.454409,-0.022993,-0.024162,0.001169,4.283632,4.384416,-0.154994,-0.113574,-0.041420,False,False,True,True,False,True
2026-09-30,4.16,4.192573,4.211469,4.446661,-0.030768,-0.025483,-0.005285,4.283632,4.384416,-0.154994,-0.113574,-0.041420,False,False,False,True,True,True
2026-10-01,4.14,4.175048,4.198474,4.438591,-0.037745,-0.027935,-0.009810,4.283632,4.384416,-0.154994,-0.113574,-0.041420,False,False,False,False,False,True
2026-10-02,4.14,4.163366,4.187843,4.430733,-0.038964,-0.030141,-0.008823,4.283632,4.384416,-0.154994,-0.113574,-0.041420,False,False,False,False,False,True
2026-10-05,4.20,4.175577,4.190053,4.424662,-0.022416,-0.028596,0.006180,4.235754,4.339977,-0.160818,-0.123023,-0.037795,False,False,True,True,True,True
2026-10-06,4.18,4.177051,4.188225,4.418223,-0.017559,-0.026389,0.008830,4.235754,4.339977,-0.160818,-0.123023,-0.037795,False,False,True,True,False,True


In [ ]:

# ============================================================
# 7) LEGACY SCORE
# ============================================================
#
# วัตถุประสงค์:
# - เก็บสูตร FAST HYBRID PRO เดิมไว้เป็น Baseline
# - ใช้เปรียบเทียบกับ Canonical Research Score ใน Cell 8
# - ยังไม่ทำ Backtest และยังไม่เปิด Position
#
# LEGACY 5-POINT SCORE
# 1. MACD 6/13/5 + Histogram Rising
# 2. Stochastic 8/3/3 : K > D และ K < 80
# 3. Trend: Close > EMA20 > EMA50
# 4. Candle Body / ATR >= 0.8
# 5. Volatility: BB breakout OR squeeze release
#
# หมายเหตุ:
# - Cell 4 ใช้ EMA5/10/75 และ MACD 4/12/9 เป็น canonical data
# - ดังนั้น Legacy MACD จะคำนวณใหม่ตรงนี้ด้วย 6/13/5
# - EMA20/EMA50 ก็จะคำนวณใหม่เพื่อรักษาสูตร Legacy เดิม
# ============================================================


def calculate_legacy_score(df):
    out = df.copy()

    # --------------------------------------------------------
    # 1) LEGACY MACD 6/13/5
    # --------------------------------------------------------
    close = out["Close"]

    legacy_ema_fast = close.ewm(
        span=6,
        adjust=False
    ).mean()

    legacy_ema_slow = close.ewm(
        span=13,
        adjust=False
    ).mean()

    out["LEGACY_MACD"] = (
        legacy_ema_fast - legacy_ema_slow
    )

    out["LEGACY_MACD_SIGNAL"] = (
        out["LEGACY_MACD"]
        .ewm(span=5, adjust=False)
        .mean()
    )

    out["LEGACY_MACD_HIST"] = (
        out["LEGACY_MACD"]
        - out["LEGACY_MACD_SIGNAL"]
    )

    out["LEGACY_MACD_HIST_PREV"] = (
        out["LEGACY_MACD_HIST"].shift(1)
    )

    out["LEGACY_MACD_BULL"] = (
        (out["LEGACY_MACD"] > out["LEGACY_MACD_SIGNAL"])
        &
        (
            out["LEGACY_MACD_HIST"]
            > out["LEGACY_MACD_HIST_PREV"]
        )
    )


    # --------------------------------------------------------
    # 2) LEGACY STOCHASTIC 8/3/3
    # --------------------------------------------------------
    lowest_8 = out["Low"].rolling(
        window=8,
        min_periods=8
    ).min()

    highest_8 = out["High"].rolling(
        window=8,
        min_periods=8
    ).max()

    price_range = (
        highest_8 - lowest_8
    ).replace(0, np.nan)

    stoch_raw = (
        100
        * (close - lowest_8)
        / price_range
    )

    out["LEGACY_STOCH_K"] = (
        stoch_raw
        .rolling(window=3, min_periods=3)
        .mean()
    )

    out["LEGACY_STOCH_D"] = (
        out["LEGACY_STOCH_K"]
        .rolling(window=3, min_periods=3)
        .mean()
    )

    out["LEGACY_STOCH_BULL"] = (
        (out["LEGACY_STOCH_K"] > out["LEGACY_STOCH_D"])
        &
        (out["LEGACY_STOCH_K"] < 80)
    )


    # --------------------------------------------------------
    # 3) LEGACY TREND
    # Close > EMA20 > EMA50
    # --------------------------------------------------------
    out["LEGACY_EMA20"] = (
        close.ewm(
            span=20,
            adjust=False
        ).mean()
    )

    out["LEGACY_EMA50"] = (
        close.ewm(
            span=50,
            adjust=False
        ).mean()
    )

    out["LEGACY_TREND_BULL"] = (
        (close > out["LEGACY_EMA20"])
        &
        (out["LEGACY_EMA20"] > out["LEGACY_EMA50"])
    )


    # --------------------------------------------------------
    # 4) CANDLE BODY / ATR >= 0.8
    # --------------------------------------------------------
    #
    # ใช้ ATR14 จาก Cell 4
    # Body = abs(Close - Open)
    #
    # เงื่อนไขเดิม:
    # Body / ATR >= 0.8
    # --------------------------------------------------------

    out["LEGACY_BODY_ATR_RATIO"] = (
        (out["Close"] - out["Open"]).abs()
        / out["ATR14"].replace(0, np.nan)
    )

    out["LEGACY_CANDLE_BULL"] = (
        out["LEGACY_BODY_ATR_RATIO"] >= 0.8
    )


    # --------------------------------------------------------
    # 5) VOLATILITY
    # BB Breakout OR Squeeze Release
    # --------------------------------------------------------
    #
    # BB breakout:
    # Close > Upper BB
    #
    # Squeeze release:
    # SQUEEZE วันนี้ False
    # และเมื่อวานเป็น True
    # --------------------------------------------------------

    out["LEGACY_BB_BREAKOUT"] = (
        out["Close"] > out["BB_UPPER"]
    )

    out["LEGACY_SQUEEZE_RELEASE"] = (
        (~out["SQUEEZE"])
        &
        (out["SQUEEZE"].shift(1).fillna(False))
    )

    out["LEGACY_VOLATILITY_BULL"] = (
        out["LEGACY_BB_BREAKOUT"]
        |
        out["LEGACY_SQUEEZE_RELEASE"]
    )


    # --------------------------------------------------------
    # LEGACY SCORE
    # --------------------------------------------------------

    score_components = [
        "LEGACY_MACD_BULL",
        "LEGACY_STOCH_BULL",
        "LEGACY_TREND_BULL",
        "LEGACY_CANDLE_BULL",
        "LEGACY_VOLATILITY_BULL",
    ]

    out["LEGACY_SCORE"] = (
        out[score_components]
        .fillna(False)
        .astype(int)
        .sum(axis=1)
    )


    # --------------------------------------------------------
    # Signal classification
    # --------------------------------------------------------
    #
    # Legacy เดิมใช้ 5 คะแนนเต็ม
    #
    # >= 3 = ผ่าน baseline
    # < 3  = ไม่ผ่าน
    # --------------------------------------------------------

    out["LEGACY_SIGNAL"] = (
        out["LEGACY_SCORE"] >= 3
    )


    return out


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

legacy_data = {}
legacy_errors = {}

print("🧮 Calculating Legacy Score...\n")

for symbol, df in feature_data.items():
    try:
        result = calculate_legacy_score(df)

        legacy_data[symbol] = result

        print(
            f"✅ {symbol:<12} "
            f"{len(result):>5} bars"
        )

    except Exception as e:
        legacy_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print(
    f"Legacy datasets: "
    f"{len(legacy_data)} / {len(feature_data)}"
)
print("====================================")


if legacy_errors:
    print("\n⚠️ Legacy calculation errors:")

    for symbol, error in legacy_errors.items():
        print(f"{symbol}: {error}")


# ============================================================
# VALIDATION
# ============================================================

if legacy_data:

    sample_symbol = next(iter(legacy_data))
    sample = legacy_data[sample_symbol]

    print(
        f"\n🔎 Legacy validation: "
        f"{sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "LEGACY_EMA20",
                "LEGACY_EMA50",

                "LEGACY_MACD",
                "LEGACY_MACD_SIGNAL",
                "LEGACY_MACD_HIST",

                "LEGACY_STOCH_K",
                "LEGACY_STOCH_D",

                "LEGACY_BODY_ATR_RATIO",

                "LEGACY_BB_BREAKOUT",
                "LEGACY_SQUEEZE_RELEASE",

                "LEGACY_MACD_BULL",
                "LEGACY_STOCH_BULL",
                "LEGACY_TREND_BULL",
                "LEGACY_CANDLE_BULL",
                "LEGACY_VOLATILITY_BULL",

                "LEGACY_SCORE",
                "LEGACY_SIGNAL",
            ]
        ].tail(10)
    )


# ============================================================
# SCORE DISTRIBUTION
# ============================================================

if legacy_data:

    all_scores = pd.concat(
        [
            df["LEGACY_SCORE"]
            for df in legacy_data.values()
        ]
    )

    print("\n📊 Legacy Score Distribution")

    display(
        all_scores
        .value_counts()
        .sort_index()
        .rename("bars")
        .to_frame()
    )

    print(
        f"\nBars with Legacy Score >= 3: "
        f"{int((all_scores >= 3).sum()):,}"
    )

    print(
        f"Total usable bars: "
        f"{len(all_scores):,}"
    )

    print(
        f"Signal rate: "
        f"{(all_scores >= 3).mean() * 100:.2f}%"
    )

🧮 Calculating Legacy Score...

✅ AU.BK         1206 bars
✅ CBG.BK        1206 bars
✅ DELTA.BK      1206 bars
✅ DITTO.BK      1206 bars
✅ FORTH.BK      1206 bars


✅ ICHI.BK       1206 bars
✅ JMART.BK      1206 bars
✅ JMT.BK        1206 bars
✅ KAMART.BK     1205 bars
✅ KLINIQ.BK      946 bars
✅ MASTER.BK      892 bars
✅ MEB.BK         878 bars
✅ MOSHI.BK       915 bars
✅ PLANB.BK      1206 bars
✅ SAPPE.BK      1206 bars


✅ SNNP.BK       1206 bars
✅ SPA.BK        1206 bars
✅ TKN.BK        1206 bars
✅ XO.BK         1206 bars

Legacy datasets: 19 / 19

🔎 Legacy validation: AU.BK


Price,Close,LEGACY_EMA20,LEGACY_EMA50,LEGACY_MACD,LEGACY_MACD_SIGNAL,LEGACY_MACD_HIST,LEGACY_STOCH_K,LEGACY_STOCH_D,LEGACY_BODY_ATR_RATIO,LEGACY_BB_BREAKOUT,LEGACY_SQUEEZE_RELEASE,LEGACY_MACD_BULL,LEGACY_STOCH_BULL,LEGACY_TREND_BULL,LEGACY_CANDLE_BULL,LEGACY_VOLATILITY_BULL,LEGACY_SCORE,LEGACY_SIGNAL
Date,,,,,,,,,,,,,,,,,,
2026-09-24,4.22,4.279364,4.403263,-0.018971,-0.020639,0.001668,19.444306,24.999807,0.000000,False,False,False,False,False,False,False,0,False
2026-09-25,4.20,4.271805,4.395292,-0.020852,-0.020710,-0.000142,8.333284,16.666545,0.463176,False,False,False,False,False,False,False,0,False
2026-09-28,4.20,4.264967,4.387634,-0.021152,-0.020857,-0.000295,21.666553,16.481381,0.000000,False,False,False,True,False,False,False,1,False
2026-09-29,4.20,4.258780,4.380275,-0.020472,-0.020729,0.000257,26.666540,18.888792,0.469043,False,False,True,True,False,False,False,2,False
2026-09-30,4.16,4.249372,4.371637,-0.024935,-0.022131,-0.002804,26.666540,24.999877,0.942252,False,False,False,True,False,True,False,2,False
2026-10-01,4.14,4.238956,4.362553,-0.029506,-0.024589,-0.004917,21.666578,24.999886,0.416748,False,True,False,False,False,False,True,1,False
2026-10-02,4.14,4.229531,4.353826,-0.031101,-0.026760,-0.004341,16.666617,21.666578,0.000000,False,False,False,False,False,False,False,0,False
2026-10-05,4.20,4.226719,4.347793,-0.022236,-0.025252,0.003016,40.476141,26.269779,0.725226,False,False,True,True,False,False,False,2,False
2026-10-06,4.18,4.222269,4.341213,-0.018759,-0.023088,0.004329,51.190451,36.111070,0.369872,False,True,True,True,False,False,True,3,True



📊 Legacy Score Distribution


,bars
LEGACY_SCORE,
0,6023
1,8135
2,5172
3,1660
4,580
5,150



Bars with Legacy Score >= 3: 2,390
Total usable bars: 21,720
Signal rate: 11.00%


In [ ]:

# ============================================================
# 8) CANONICAL RESEARCH SCORE
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
# สร้างคะแนนระบบใหม่เพื่อใช้เปรียบเทียบกับ Legacy Score
#
# CANONICAL INDICATORS
#   DAY MACD      = 4 / 12 / 9
#   STOCHASTIC    = 8 / 3 / 3
#   DAY EMA       = 5 / 10 / 75
#   WEEK EMA      = 5 / 10
#   WEEK MACD     = 4 / 12 / 9
#
# RESEARCH SCORE = 5 POINTS
#
# 1) MACD
#    DAY MACD > Signal
#    AND Histogram > 0
#
# 2) STOCHASTIC
#    K > D
#    AND K < 80
#
# 3) DAY TREND
#    Close > EMA5 > EMA10 > EMA75
#
# 4) EARLY ENTRY
#    Squeeze OR Squeeze Release
#
# 5) VOLUME
#    Volume Ratio >= 1.2
#
# PASS:
#    Score >= 3
#
# IMPORTANT
# ------------------------------------------------------------
# Cell นี้เป็น Research Baseline
# ไม่ใช่ Production Weighted Score
# ============================================================


def calculate_canonical_research_score(df):

    out = df.copy()

    # --------------------------------------------------------
    # 1) DAY MACD 4/12/9
    # --------------------------------------------------------

    out["CANONICAL_MACD_BULL"] = (
        (out["MACD"] > out["MACD_SIGNAL"])
        &
        (out["MACD_HIST"] > 0)
    )


    # --------------------------------------------------------
    # 2) STOCHASTIC 8/3/3
    # --------------------------------------------------------

    out["CANONICAL_STOCH_BULL"] = (
        (out["STOCH_K"] > out["STOCH_D"])
        &
        (out["STOCH_K"] < 80)
    )


    # --------------------------------------------------------
    # 3) DAY TREND
    #
    # Close > EMA5 > EMA10 > EMA75
    # --------------------------------------------------------

    out["CANONICAL_TREND_BULL"] = (
        (out["Close"] > out["EMA5"])
        &
        (out["EMA5"] > out["EMA10"])
        &
        (out["EMA10"] > out["EMA75"])
    )


    # --------------------------------------------------------
    # 4) EARLY ENTRY
    #
    # ใช้ Squeeze หรือ Squeeze Release
    #
    # ไม่ใช้ SQUEEZE_RECENT ในคะแนนหลัก
    # เพื่อไม่ให้ signal window กว้างเกินไป
    # --------------------------------------------------------

    out["CANONICAL_EARLY_ENTRY"] = (
        out["SQUEEZE"]
        |
        out["SQUEEZE_RELEASE"]
    )


    # --------------------------------------------------------
    # 5) VOLUME CONFIRMATION
    # --------------------------------------------------------

    out["CANONICAL_VOLUME_CONFIRM"] = (
        out["VOLUME_RATIO"] >= 1.2
    )


    # --------------------------------------------------------
    # SCORE COMPONENTS
    # --------------------------------------------------------

    canonical_components = [
        "CANONICAL_MACD_BULL",
        "CANONICAL_STOCH_BULL",
        "CANONICAL_TREND_BULL",
        "CANONICAL_EARLY_ENTRY",
        "CANONICAL_VOLUME_CONFIRM",
    ]

    out["CANONICAL_SCORE"] = (
        out[canonical_components]
        .fillna(False)
        .astype(int)
        .sum(axis=1)
    )


    # --------------------------------------------------------
    # SIGNAL
    # --------------------------------------------------------

    out["CANONICAL_SIGNAL"] = (
        out["CANONICAL_SCORE"] >= 3
    )


    # --------------------------------------------------------
    # ADDITIONAL INFORMATION
    # --------------------------------------------------------
    #
    # Weekly confirmation ไม่ถูกนับเป็นคะแนนเพิ่ม
    # ใน Research Score
    #
    # เก็บไว้เพื่อใช้วิเคราะห์ภายหลัง
    # --------------------------------------------------------

    out["CANONICAL_WEEKLY_TREND"] = (
        out["W_EMA5"] > out["W_EMA10"]
    )

    out["CANONICAL_WEEKLY_MACD"] = (
        out["W_MACD"] > out["W_MACD_SIGNAL"]
    )

    out["CANONICAL_WEEKLY_CONFIRM"] = (
        out["CANONICAL_WEEKLY_TREND"]
        &
        out["CANONICAL_WEEKLY_MACD"]
    )


    return out


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

canonical_data = {}
canonical_errors = {}

print("🧮 Calculating Canonical Research Score...\n")

for symbol, df in feature_data.items():

    try:

        result = calculate_canonical_research_score(df)

        canonical_data[symbol] = result

        print(
            f"✅ {symbol:<12} "
            f"{len(result):>5} bars"
        )

    except Exception as e:

        canonical_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print(
    f"Canonical datasets: "
    f"{len(canonical_data)} / {len(feature_data)}"
)
print("====================================")


if canonical_errors:

    print("\n⚠️ Canonical calculation errors:")

    for symbol, error in canonical_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# VALIDATION
# ============================================================

if canonical_data:

    sample_symbol = next(iter(canonical_data))

    sample = canonical_data[sample_symbol]

    print(
        f"\n🔎 Canonical validation: "
        f"{sample_symbol}"
    )

    display(
        sample[
            [
                "Close",

                "EMA5",
                "EMA10",
                "EMA75",

                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",

                "STOCH_K",
                "STOCH_D",

                "W_EMA5",
                "W_EMA10",

                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST",

                "VOLUME_RATIO",

                "SQUEEZE",
                "SQUEEZE_RELEASE",

                "CANONICAL_MACD_BULL",
                "CANONICAL_STOCH_BULL",
                "CANONICAL_TREND_BULL",
                "CANONICAL_EARLY_ENTRY",
                "CANONICAL_VOLUME_CONFIRM",

                "CANONICAL_SCORE",
                "CANONICAL_SIGNAL",

                "CANONICAL_WEEKLY_TREND",
                "CANONICAL_WEEKLY_MACD",
                "CANONICAL_WEEKLY_CONFIRM",
            ]
        ].tail(10)
    )


# ============================================================
# SCORE DISTRIBUTION
# ============================================================

if canonical_data:

    all_canonical_scores = pd.concat(
        [
            df["CANONICAL_SCORE"]
            for df in canonical_data.values()
        ]
    )

    print("\n📊 Canonical Score Distribution")

    display(
        all_canonical_scores
        .value_counts()
        .sort_index()
        .rename("bars")
        .to_frame()
    )


    signal_count = int(
        (all_canonical_scores >= 3).sum()
    )

    total_bars = len(
        all_canonical_scores
    )

    signal_rate = (
        signal_count / total_bars * 100
        if total_bars > 0
        else 0
    )


    print(
        f"\nBars with Canonical Score >= 3: "
        f"{signal_count:,}"
    )

    print(
        f"Total usable bars: "
        f"{total_bars:,}"
    )

    print(
        f"Canonical signal rate: "
        f"{signal_rate:.2f}%"
    )


# ============================================================
# LEGACY vs CANONICAL QUICK COMPARISON
# ============================================================

if legacy_data and canonical_data:

    legacy_scores = pd.concat(
        [
            df["LEGACY_SCORE"]
            for df in legacy_data.values()
        ]
    )

    canonical_scores = pd.concat(
        [
            df["CANONICAL_SCORE"]
            for df in canonical_data.values()
        ]
    )

    print("\n====================================")
    print("LEGACY vs CANONICAL")
    print("====================================")

    print(
        f"Legacy signal rate:     "
        f"{(legacy_scores >= 3).mean() * 100:.2f}%"
    )

    print(
        f"Canonical signal rate:  "
        f"{(canonical_scores >= 3).mean() * 100:.2f}%"
    )

🧮 Calculating Canonical Research Score...

✅ AU.BK         1206 bars
✅ CBG.BK        1206 bars


✅ DELTA.BK      1206 bars
✅ DITTO.BK      1206 bars
✅ FORTH.BK      1206 bars
✅ ICHI.BK       1206 bars
✅ JMART.BK      1206 bars
✅ JMT.BK        1206 bars
✅ KAMART.BK     1205 bars
✅ KLINIQ.BK      946 bars


✅ MASTER.BK      892 bars


✅ MEB.BK         878 bars
✅ MOSHI.BK       915 bars
✅ PLANB.BK      1206 bars
✅ SAPPE.BK      1206 bars
✅ SNNP.BK       1206 bars
✅ SPA.BK        1206 bars
✅ TKN.BK        1206 bars
✅ XO.BK         1206 bars

Canonical datasets: 19 / 19

🔎 Canonical validation: AU.BK


Price,Close,EMA5,EMA10,EMA75,MACD,MACD_SIGNAL,MACD_HIST,STOCH_K,STOCH_D,W_EMA5,W_EMA10,W_MACD,W_MACD_SIGNAL,W_MACD_HIST,VOLUME_RATIO,SQUEEZE,SQUEEZE_RELEASE,CANONICAL_MACD_BULL,CANONICAL_STOCH_BULL,CANONICAL_TREND_BULL,CANONICAL_EARLY_ENTRY,CANONICAL_VOLUME_CONFIRM,CANONICAL_SCORE,CANONICAL_SIGNAL,CANONICAL_WEEKLY_TREND,CANONICAL_WEEKLY_MACD,CANONICAL_WEEKLY_CONFIRM
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-09-24,4.22,4.229900,4.241822,4.475599,-0.020207,-0.024648,0.004441,19.444306,24.999807,4.325448,4.425397,-0.153826,-0.103219,-0.050607,1.484927,False,False,True,False,False,False,True,2,False,False,False,False
2026-09-25,4.20,4.219933,4.234218,4.468346,-0.023887,-0.024496,0.000609,8.333284,16.666545,4.325448,4.425397,-0.153826,-0.103219,-0.050607,0.959982,False,False,True,False,False,False,False,1,False,False,False,False
2026-09-28,4.20,4.213289,4.227997,4.461285,-0.024285,-0.024454,0.000169,21.666553,16.481381,4.283632,4.384416,-0.154994,-0.113574,-0.041420,1.321540,True,False,True,True,False,True,True,4,True,False,False,False
2026-09-29,4.20,4.208859,4.222906,4.454409,-0.022993,-0.024162,0.001169,26.666540,18.888792,4.283632,4.384416,-0.154994,-0.113574,-0.041420,0.534683,True,False,True,True,False,True,False,3,True,False,False,False
2026-09-30,4.16,4.192573,4.211469,4.446661,-0.030768,-0.025483,-0.005285,26.666540,24.999877,4.283632,4.384416,-0.154994,-0.113574,-0.041420,1.524227,True,False,False,True,False,True,True,3,True,False,False,False
2026-10-01,4.14,4.175048,4.198474,4.438591,-0.037745,-0.027935,-0.009810,21.666578,24.999886,4.283632,4.384416,-0.154994,-0.113574,-0.041420,1.063792,False,True,False,False,False,True,False,1,False,False,False,False
2026-10-02,4.14,4.163366,4.187843,4.430733,-0.038964,-0.030141,-0.008823,16.666617,21.666578,4.283632,4.384416,-0.154994,-0.113574,-0.041420,1.187989,False,False,False,False,False,False,False,0,False,False,False,False
2026-10-05,4.20,4.175577,4.190053,4.424662,-0.022416,-0.028596,0.006180,40.476141,26.269779,4.235754,4.339977,-0.160818,-0.123023,-0.037795,1.358750,True,False,True,True,False,True,True,4,True,False,False,False
2026-10-06,4.18,4.177051,4.188225,4.418223,-0.017559,-0.026389,0.008830,51.190451,36.111070,4.235754,4.339977,-0.160818,-0.123023,-0.037795,1.037210,False,True,True,True,False,True,False,3,True,False,False,False



📊 Canonical Score Distribution


,bars
CANONICAL_SCORE,
0,3580
1,7198
2,6709
3,3031
4,981
5,221



Bars with Canonical Score >= 3: 4,233
Total usable bars: 21,720
Canonical signal rate: 19.49%

LEGACY vs CANONICAL
Legacy signal rate:     11.00%
Canonical signal rate:  19.49%


In [ ]:

# ============================================================
# 9) MONEY MANAGEMENT — FINAL
# ============================================================

INITIAL_CAPITAL = 100_000.0
MAX_RISK_PCT = 0.02
RISK_PER_TRADE = INITIAL_CAPITAL * MAX_RISK_PCT

LOT_SIZE = 100
ATR_STOP_MULT = 1.5
RECENT_LOW_LOOKBACK = 5


print("💰 Money Management Configuration")
print("------------------------------------")
print(f"Initial Capital : {INITIAL_CAPITAL:,.2f} THB")
print(f"Risk / Trade    : {MAX_RISK_PCT * 100:.2f}%")
print(f"Risk Amount     : {RISK_PER_TRADE:,.2f} THB")
print(f"Lot Size        : {LOT_SIZE:,} shares")
print(f"ATR Stop Mult   : {ATR_STOP_MULT:.2f}x")
print(f"Recent Low      : {RECENT_LOW_LOOKBACK} bars")


# ============================================================
# STOP PRICE
# ============================================================

def calculate_stop_price(
    entry_price,
    atr,
    recent_low,
    atr_multiplier=ATR_STOP_MULT,
):
    if pd.isna(entry_price) or entry_price <= 0:
        return np.nan

    if pd.isna(atr) or atr <= 0:
        return np.nan

    if pd.isna(recent_low) or recent_low <= 0:
        return np.nan

    atr_stop = entry_price - atr * atr_multiplier

    # Conservative stop:
    # choose the LOWER of ATR stop and recent low
    stop_price = min(atr_stop, recent_low)

    if stop_price <= 0:
        return np.nan

    if stop_price >= entry_price:
        return np.nan

    return float(stop_price)


# ============================================================
# POSITION SIZE
# ============================================================

def calculate_position_size(
    capital,
    entry_price,
    stop_price,
    risk_pct=MAX_RISK_PCT,
    lot_size=LOT_SIZE,
):
    if pd.isna(capital) or capital <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "INVALID_CAPITAL",
        }

    if pd.isna(entry_price) or entry_price <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "INVALID_ENTRY",
        }

    if pd.isna(stop_price) or stop_price <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "INVALID_STOP",
        }

    if stop_price >= entry_price:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "STOP_NOT_BELOW_ENTRY",
        }

    max_risk = capital * risk_pct
    risk_per_share = entry_price - stop_price

    if risk_per_share <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": float(risk_per_share),
            "position_value": 0.0,
            "status": "INVALID_RISK_PER_SHARE",
        }

    # --------------------------------------------------------
    # Limit #1: risk-based position size
    # --------------------------------------------------------

    risk_lots = int(
        (max_risk / risk_per_share) // lot_size
    )

    # --------------------------------------------------------
    # Limit #2: capital-based position size
    # --------------------------------------------------------

    capital_lots = int(
        (capital / entry_price) // lot_size
    )

    # Final lots must satisfy BOTH constraints
    lots = min(risk_lots, capital_lots)

    shares = lots * lot_size

    if shares < lot_size:
        if capital_lots < 1:
            status = "INSUFFICIENT_CAPITAL_FOR_ONE_LOT"
        else:
            status = "ONE_LOT_EXCEEDS_RISK"

        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": float(risk_per_share),
            "position_value": 0.0,
            "status": status,
        }

    actual_risk = shares * risk_per_share
    position_value = shares * entry_price

    if capital_lots < risk_lots:
        status = "CAPITAL_LIMITED"
    else:
        status = "OK"

    return {
        "shares": int(shares),
        "lots": int(lots),
        "risk_amount": float(actual_risk),
        "risk_per_share": float(risk_per_share),
        "position_value": float(position_value),
        "status": status,
    }


# ============================================================
# COMPLETE TRADE PLAN
# ============================================================

def build_trade_plan(
    capital,
    entry_price,
    atr,
    recent_low,
    risk_pct=MAX_RISK_PCT,
    atr_multiplier=ATR_STOP_MULT,
    lot_size=LOT_SIZE,
):
    stop_price = calculate_stop_price(
        entry_price,
        atr,
        recent_low,
        atr_multiplier,
    )

    if pd.isna(stop_price):
        return {
            "entry_price": (
                float(entry_price)
                if pd.notna(entry_price)
                else np.nan
            ),
            "stop_price": np.nan,
            "shares": 0,
            "lots": 0,
            "risk_per_share": np.nan,
            "risk_amount": 0.0,
            "position_value": 0.0,
            "status": "INVALID_STOP",
        }

    position = calculate_position_size(
        capital,
        entry_price,
        stop_price,
        risk_pct,
        lot_size,
    )

    return {
        "entry_price": float(entry_price),
        "stop_price": float(stop_price),
        "shares": position["shares"],
        "lots": position["lots"],
        "risk_per_share": position["risk_per_share"],
        "risk_amount": position["risk_amount"],
        "position_value": position["position_value"],
        "status": position["status"],
    }


# ============================================================
# RECENT LOW
# ============================================================

for symbol, df in canonical_data.items():
    df["RECENT_LOW_5"] = (
        df["Low"]
        .rolling(
            window=RECENT_LOW_LOOKBACK,
            min_periods=RECENT_LOW_LOOKBACK,
        )
        .min()
    )


# ============================================================
# VALIDATION
# ============================================================

print("\n====================================")
print("🧪 Money Management Validation")
print("====================================")


# ------------------------------------------------------------
# Test 1: Normal trade
# ------------------------------------------------------------

test_entry = 50.00
test_atr = 1.50
test_recent_low = 47.00

test_plan = build_trade_plan(
    INITIAL_CAPITAL,
    test_entry,
    test_atr,
    test_recent_low,
)

print("\nTest 1: Normal Trade")
print("------------------------------------")

for key, value in test_plan.items():
    if isinstance(value, float):
        print(f"{key:<20}: {value:,.4f}")
    else:
        print(f"{key:<20}: {value}")


# ------------------------------------------------------------
# Test 2: Stop validation
# ------------------------------------------------------------

test_entry = 10.00
test_atr = 0.50
test_recent_low = 9.80

test_plan_2 = build_trade_plan(
    INITIAL_CAPITAL,
    test_entry,
    test_atr,
    test_recent_low,
)

print("\nTest 2: Stop Validation")
print("------------------------------------")
print(f"Entry : {test_plan_2['entry_price']:.4f}")
print(f"Stop  : {test_plan_2['stop_price']:.4f}")

if pd.notna(test_plan_2["stop_price"]):
    print(
        "Valid : "
        f"{test_plan_2['stop_price'] < test_plan_2['entry_price']}"
    )
else:
    print("Valid : False")


# ------------------------------------------------------------
# Test 3: One lot risk check
# ------------------------------------------------------------

test_entry = 80.00
test_stop = 60.00

test_position = calculate_position_size(
    INITIAL_CAPITAL,
    test_entry,
    test_stop,
)

print("\nTest 3: One Lot Risk Check")
print("------------------------------------")
print(f"Entry       : {test_entry:.2f}")
print(f"Stop        : {test_stop:.2f}")
print(f"Risk / Share: {test_position['risk_per_share']:.2f}")
print(f"Shares      : {test_position['shares']:,}")
print(f"Lots        : {test_position['lots']:,}")
print(f"Risk Amount : {test_position['risk_amount']:,.2f}")
print(f"Position    : {test_position['position_value']:,.2f}")
print(f"Status      : {test_position['status']}")


# ------------------------------------------------------------
# Test 4: Capital limit
# ------------------------------------------------------------

test_entry = 1.00
test_stop = 0.99

test_position_4 = calculate_position_size(
    INITIAL_CAPITAL,
    test_entry,
    test_stop,
)

print("\nTest 4: Capital Limit Check")
print("------------------------------------")
print(f"Entry       : {test_entry:.2f}")
print(f"Stop        : {test_stop:.2f}")
print(f"Risk / Share: {test_position_4['risk_per_share']:.4f}")
print(f"Shares      : {test_position_4['shares']:,}")
print(f"Lots        : {test_position_4['lots']:,}")
print(f"Risk Amount : {test_position_4['risk_amount']:,.2f}")
print(f"Position    : {test_position_4['position_value']:,.2f}")
print(f"Status      : {test_position_4['status']}")

assert (
    test_position_4["position_value"]
    <= INITIAL_CAPITAL + 1e-9
), "Position value exceeds capital"


# ============================================================
# REAL DATA SANITY CHECK
# ============================================================

print("\n====================================")
print("🔎 Real Data Sanity Check")
print("====================================")

if canonical_data:
    sample_symbol = next(iter(canonical_data))
    sample = canonical_data[sample_symbol]

    valid_rows = (
        sample[
            ["Close", "ATR14", "RECENT_LOW_5"]
        ]
        .dropna()
        .tail(3)
    )

    print(f"\nSample: {sample_symbol}")
    display(valid_rows)


# ============================================================
# FINAL CHECKS
# ============================================================

assert INITIAL_CAPITAL > 0
assert 0 < MAX_RISK_PCT <= 0.05
assert LOT_SIZE > 0
assert ATR_STOP_MULT > 0
assert RECENT_LOW_LOOKBACK > 0

print("\n====================================")
print("✅ Money Management Cell Ready")
print("====================================")
print(f"Maximum risk per trade: {RISK_PER_TRADE:,.2f} THB")
print("Stop rule: min(Entry - ATR × 1.5, Recent Low)")
print("Position size: whole 100-share lots, rounded DOWN")
print("Position limit: Risk limit + Capital limit")
print("Target prices: calculated later from actual Entry")

💰 Money Management Configuration
------------------------------------
Initial Capital : 100,000.00 THB
Risk / Trade    : 2.00%
Risk Amount     : 2,000.00 THB
Lot Size        : 100 shares
ATR Stop Mult   : 1.50x
Recent Low      : 5 bars



🧪 Money Management Validation

Test 1: Normal Trade
------------------------------------
entry_price         : 50.0000
stop_price          : 47.0000
shares              : 600
lots                : 6
risk_per_share      : 3.0000
risk_amount         : 1,800.0000
position_value      : 30,000.0000
status              : OK

Test 2: Stop Validation
------------------------------------
Entry : 10.0000
Stop  : 9.2500
Valid : True

Test 3: One Lot Risk Check
------------------------------------
Entry       : 80.00
Stop        : 60.00
Risk / Share: 20.00
Shares      : 100
Lots        : 1
Risk Amount : 2,000.00
Position    : 8,000.00
Status      : OK

Test 4: Capital Limit Check
------------------------------------
Entry       : 1.00
Stop        : 0.99
Risk / Share: 0.0100
Shares      : 100,000
Lots        : 1,000
Risk Amount : 1,000.00
Position    : 100,000.00
Status      : CAPITAL_LIMITED

🔎 Real Data Sanity Check

Sample: AU.BK


Price,Close,ATR14,RECENT_LOW_5
Date,,,
2026-10-05,4.20,0.055155,4.1
2026-10-06,4.18,0.054073,4.1
2026-10-07,4.16,0.054496,4.1



✅ Money Management Cell Ready
Maximum risk per trade: 2,000.00 THB
Stop rule: min(Entry - ATR × 1.5, Recent Low)
Position size: whole 100-share lots, rounded DOWN
Position limit: Risk limit + Capital limit
Target prices: calculated later from actual Entry


In [ ]:

# ============================================================
# 10) CURRENT SCANNER
# ============================================================

print("🔎 CURRENT SCANNER")
print("=" * 90)


def build_current_scan_row(symbol, df):
    if df is None or df.empty:
        return {
            "Symbol": symbol,
            "Date": pd.NaT,
            "Close": np.nan,
            "Score": np.nan,
            "Signal": False,
            "WeeklyConfirm": False,
            "MACD": np.nan,
            "MACD_Signal": np.nan,
            "MACD_Hist": np.nan,
            "EMA5": np.nan,
            "EMA10": np.nan,
            "EMA75": np.nan,
            "W_EMA5": np.nan,
            "W_EMA10": np.nan,
            "W_MACD": np.nan,
            "W_MACD_Signal": np.nan,
            "ATR14": np.nan,
            "RecentLow5": np.nan,
            "Stop": np.nan,
            "RiskPerShare": np.nan,
            "Shares": 0,
            "Lots": 0,
            "RiskAmount": 0.0,
            "PositionValue": 0.0,
            "Target1": np.nan,
            "Target2": np.nan,
            "Status": "NO_DATA",
        }

    row = df.iloc[-1]

    entry_price = row["Close"]
    atr = row["ATR14"]
    recent_low = row["RECENT_LOW_5"]

    trade_plan = build_trade_plan(
        INITIAL_CAPITAL,
        entry_price,
        atr,
        recent_low,
        atr_multiplier=PRODUCTION_ATR_MULT,
    )

    score = row.get("CANONICAL_SCORE", np.nan)
    signal = (
        symbol in PRODUCTION_SYMBOLS
        and float(row.get("CANONICAL_SCORE", 0)) >= PRODUCTION_MIN_SCORE
        and (
            not PRODUCTION_WEEKLY_CONFIRM
            or bool(row.get("CANONICAL_WEEKLY_CONFIRM", False))
        )
    )

    weekly_trend = bool(
        row.get("CANONICAL_WEEKLY_TREND", False)
    )

    weekly_macd = bool(
        row.get("CANONICAL_WEEKLY_MACD", False)
    )

    weekly_confirm = bool(
        row.get("CANONICAL_WEEKLY_CONFIRM", False)
    )

    target1 = (
        float(entry_price * 1.07)
        if pd.notna(entry_price)
        else np.nan
    )

    target2 = (
        float(entry_price * (1 + PRODUCTION_TARGET2_PCT))
        if pd.notna(entry_price)
        else np.nan
    )

    return {
        "Symbol": symbol,
        "Date": df.index[-1],
        "Close": float(entry_price),
        "Score": score,
        "Signal": signal,
        "WeeklyTrend": weekly_trend,
        "WeeklyMACD": weekly_macd,
        "WeeklyConfirm": weekly_confirm,
        "MACD": row["MACD"],
        "MACD_Signal": row["MACD_SIGNAL"],
        "MACD_Hist": row["MACD_HIST"],
        "EMA5": row["EMA5"],
        "EMA10": row["EMA10"],
        "EMA75": row["EMA75"],
        "W_EMA5": row["W_EMA5"],
        "W_EMA10": row["W_EMA10"],
        "W_MACD": row["W_MACD"],
        "W_MACD_Signal": row["W_MACD_SIGNAL"],
        "ATR14": row["ATR14"],
        "RecentLow5": row["RECENT_LOW_5"],
        "Stop": trade_plan["stop_price"],
        "RiskPerShare": trade_plan["risk_per_share"],
        "Shares": trade_plan["shares"],
        "Lots": trade_plan["lots"],
        "RiskAmount": trade_plan["risk_amount"],
        "PositionValue": trade_plan["position_value"],
        "Target1": target1,
        "Target2": target2,
        "Status": trade_plan["status"],
    }


# ============================================================
# BUILD CURRENT SCAN
# ============================================================

current_scan_rows = []

for symbol, df in canonical_data.items():
    current_scan_rows.append(
        build_current_scan_row(symbol, df)
    )

current_scan = pd.DataFrame(current_scan_rows)


# ============================================================
# SORT
# ============================================================

current_scan = current_scan.sort_values(
    by=["Signal", "Score", "WeeklyConfirm"],
    ascending=[False, False, False],
    na_position="last",
).reset_index(drop=True)


# ============================================================
# DISPLAY
# ============================================================

print("\n📊 Current Market Snapshot")
print("-" * 90)

display(
    current_scan[
        [
            "Symbol",
            "Date",
            "Close",
            "Score",
            "Signal",
            "WeeklyConfirm",
            "ATR14",
            "Stop",
            "Shares",
            "RiskAmount",
            "Target1",
            "Target2",
            "Status",
        ]
    ]
)


# ============================================================
# SIGNAL SUMMARY
# ============================================================

signal_scan = current_scan[
    current_scan["Signal"] == True
].copy()

weekly_confirmed = current_scan[
    current_scan["WeeklyConfirm"] == True
].copy()

print("\n====================================")
print("📈 CURRENT SCANNER SUMMARY")
print("====================================")

print(f"Total symbols       : {len(current_scan)}")
print(f"Production universe : {len(PRODUCTION_SYMBOLS)}")
print(f"Production min score: {PRODUCTION_MIN_SCORE}")
print(f"Production ATR mult : {PRODUCTION_ATR_MULT:.2f}x")
print(f"Production target2  : {PRODUCTION_TARGET2_PCT * 100:.1f}%")
print(f"Canonical signals   : {len(signal_scan)}")
print(f"Weekly confirmed    : {len(weekly_confirmed)}")

if len(current_scan) > 0:
    signal_rate = (
        len(signal_scan) / len(current_scan) * 100
    )
else:
    signal_rate = 0.0

print(f"Signal rate         : {signal_rate:.2f}%")


# ============================================================
# TOP SIGNALS
# ============================================================

print("\n====================================")
print("🏆 TOP CURRENT SIGNALS")
print("====================================")

if signal_scan.empty:
    print("No current Canonical signals.")
else:
    display(
        signal_scan[
            [
                "Symbol",
                "Date",
                "Close",
                "Score",
                "WeeklyConfirm",
                "ATR14",
                "Stop",
                "Shares",
                "RiskAmount",
                "Target1",
                "Target2",
            ]
        ].head(10)
    )


# ============================================================
# SANITY CHECKS
# ============================================================

assert len(current_scan) == len(canonical_data)

assert current_scan["Score"].dropna().between(0, 5).all()

assert (
    current_scan["RiskAmount"].fillna(0)
    <= RISK_PER_TRADE + 1e-9
).all()

assert (
    current_scan["PositionValue"].fillna(0)
    <= INITIAL_CAPITAL + 1e-9
).all()

print("\n====================================")
print("✅ Current Scanner Cell Ready")
print("====================================")

🔎 CURRENT SCANNER

📊 Current Market Snapshot
------------------------------------------------------------------------------------------


,Symbol,Date,Close,Score,Signal,WeeklyConfirm,ATR14,Stop,Shares,RiskAmount,Target1,Target2,Status
0,KLINIQ.BK,2026-10-07,29.250000,3,False,False,0.775453,27.250000,1000,2000.000000,31.297500,33.637500,OK
1,PLANB.BK,2026-10-07,7.850000,2,False,True,0.325651,6.850000,2000,2000.000000,8.399500,9.027500,OK
2,AU.BK,2026-10-07,4.160000,2,False,False,0.054496,4.094605,24000,1569.485260,4.451200,4.784000,CAPITAL_LIMITED
3,CBG.BK,2026-10-07,53.500000,2,False,False,1.636215,50.000000,500,1750.000000,57.245000,61.525000,OK
4,DITTO.BK,2026-10-07,12.300000,2,False,False,0.338452,11.893858,4900,1990.098856,13.161000,14.145000,OK
5,JMART.BK,2026-10-07,9.700000,2,False,False,0.260627,9.387247,6300,1970.339761,10.379000,11.155000,OK
6,KAMART.BK,2026-10-07,7.200000,2,False,False,0.165693,6.650000,3600,1979.998970,7.704000,8.280000,OK
7,MEB.BK,2026-10-07,10.600000,2,False,False,0.218358,10.200000,4900,1960.002804,11.342000,12.190000,OK
8,MOSHI.BK,2026-10-07,34.500000,2,False,False,0.734842,33.618190,2200,1939.982288,36.915000,39.675000,OK
9,SNNP.BK,2026-10-07,6.650000,2,False,False,0.100491,6.529411,15000,1808.841804,7.115500,7.647500,CAPITAL_LIMITED



📈 CURRENT SCANNER SUMMARY
Total symbols       : 19
Production universe : 8
Production min score: 4
Production ATR mult : 1.20x
Production target2  : 15.0%
Canonical signals   : 0
Weekly confirmed    : 2
Signal rate         : 0.00%

🏆 TOP CURRENT SIGNALS
No current Canonical signals.

✅ Current Scanner Cell Ready


In [ ]:

# ============================================================
# 11) HISTORICAL SIGNAL GENERATION
# ============================================================

print("📡 HISTORICAL SIGNAL GENERATION")
print("=" * 90)


SIGNAL_MIN_SCORE = 3


def generate_historical_signals(
    symbol,
    df,
    min_score=SIGNAL_MIN_SCORE,
):
    if df is None or df.empty:
        return pd.DataFrame()

    data = df.copy()

    # --------------------------------------------------------
    # Required columns
    # --------------------------------------------------------

    required = [
        "Close",
        "Open",
        "High",
        "Low",
        "ATR14",
        "RECENT_LOW_5",
        "CANONICAL_SCORE",
        "CANONICAL_SIGNAL",
        "CANONICAL_WEEKLY_TREND",
        "CANONICAL_WEEKLY_MACD",
        "CANONICAL_WEEKLY_CONFIRM",
    ]

    missing = [
        col for col in required
        if col not in data.columns
    ]

    if missing:
        raise ValueError(
            f"{symbol}: missing columns {missing}"
        )

    # --------------------------------------------------------
    # Signal rows
    # --------------------------------------------------------

    signal_mask = (
        data["CANONICAL_SIGNAL"].fillna(False)
        & (data["CANONICAL_SCORE"] >= min_score)
    )

    signal_dates = data.index[signal_mask]

    rows = []

    for signal_date in signal_dates:
        signal_pos = data.index.get_loc(signal_date)

        # Need next trading day for entry
        if signal_pos + 1 >= len(data):
            continue

        entry_date = data.index[signal_pos + 1]

        signal_row = data.iloc[signal_pos]
        entry_row = data.iloc[signal_pos + 1]

        signal_close = signal_row["Close"]
        signal_atr = signal_row["ATR14"]
        recent_low = signal_row["RECENT_LOW_5"]

        # ----------------------------------------------------
        # Stop is calculated from information available
        # at SIGNAL DATE.
        # ----------------------------------------------------

        stop_price = calculate_stop_price(
            signal_close,
            signal_atr,
            recent_low,
        )

        # ----------------------------------------------------
        # Entry is NEXT DAY OPEN.
        # ----------------------------------------------------

        entry_price = entry_row["Open"]

        # ----------------------------------------------------
        # If entry is invalid, skip.
        # ----------------------------------------------------

        if pd.isna(entry_price) or entry_price <= 0:
            continue

        if pd.isna(stop_price) or stop_price <= 0:
            continue

        # ----------------------------------------------------
        # Position sizing uses actual next-day entry.
        # ----------------------------------------------------

        trade_plan = build_trade_plan(
            INITIAL_CAPITAL,
            entry_price,
            signal_atr,
            recent_low,
        )

        # ----------------------------------------------------
        # Targets based on ACTUAL entry.
        # ----------------------------------------------------

        target1 = entry_price * 1.07
        target2 = entry_price * 1.15

        rows.append({
            "Symbol": symbol,

            "SignalDate": signal_date,
            "EntryDate": entry_date,

            "SignalClose": float(signal_close),
            "EntryPrice": float(entry_price),

            "Score": int(signal_row["CANONICAL_SCORE"]),

            "WeeklyTrend": bool(
                signal_row["CANONICAL_WEEKLY_TREND"]
            ),

            "WeeklyMACD": bool(
                signal_row["CANONICAL_WEEKLY_MACD"]
            ),

            "WeeklyConfirm": bool(
                signal_row["CANONICAL_WEEKLY_CONFIRM"]
            ),

            "ATR14": float(signal_atr),

            "RecentLow5": float(recent_low),

            "StopPrice": float(
                trade_plan["stop_price"]
            ),

            "RiskPerShare": float(
                trade_plan["risk_per_share"]
            ),

            "Shares": int(
                trade_plan["shares"]
            ),

            "Lots": int(
                trade_plan["lots"]
            ),

            "RiskAmount": float(
                trade_plan["risk_amount"]
            ),

            "PositionValue": float(
                trade_plan["position_value"]
            ),

            "Target1": float(target1),
            "Target2": float(target2),

            "EntryOpen": float(entry_row["Open"]),
            "EntryHigh": float(entry_row["High"]),
            "EntryLow": float(entry_row["Low"]),
            "EntryClose": float(entry_row["Close"]),
        })

    return pd.DataFrame(rows)


# ============================================================
# GENERATE SIGNALS FOR ALL SYMBOLS
# ============================================================

historical_signals_list = []

for symbol, df in canonical_data.items():

    signals = generate_historical_signals(
        symbol,
        df,
        min_score=SIGNAL_MIN_SCORE,
    )

    if not signals.empty:
        historical_signals_list.append(signals)


if historical_signals_list:
    historical_signals = pd.concat(
        historical_signals_list,
        ignore_index=True,
    )
else:
    historical_signals = pd.DataFrame()


# ============================================================
# SORT
# ============================================================

if not historical_signals.empty:
    historical_signals = historical_signals.sort_values(
        ["SignalDate", "Score"],
        ascending=[True, False],
    ).reset_index(drop=True)


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print("📊 HISTORICAL SIGNAL SUMMARY")
print("====================================")

print(
    f"Total signals : {len(historical_signals):,}"
)

if not historical_signals.empty:

    print(
        f"Symbols       : "
        f"{historical_signals['Symbol'].nunique()}"
    )

    print(
        f"Score >= 3    : "
        f"{(historical_signals['Score'] >= 3).sum():,}"
    )

    print(
        f"Weekly confirm: "
        f"{historical_signals['WeeklyConfirm'].sum():,}"
    )

    print(
        f"Date range    : "
        f"{historical_signals['SignalDate'].min().date()} "
        f"→ "
        f"{historical_signals['SignalDate'].max().date()}"
    )


# ============================================================
# LOOKAHEAD SANITY CHECK
# ============================================================

print("\n====================================")
print("🔬 LOOKAHEAD SANITY CHECK")
print("====================================")

if not historical_signals.empty:

    invalid_entry = (
        historical_signals["EntryDate"]
        <= historical_signals["SignalDate"]
    ).sum()

    invalid_price = (
        historical_signals["EntryPrice"] <= 0
    ).sum()

    invalid_stop = (
        historical_signals["StopPrice"]
        >= historical_signals["SignalClose"]
    ).sum()

    print(
        f"EntryDate <= SignalDate : {invalid_entry}"
    )

    print(
        f"Invalid EntryPrice      : {invalid_price}"
    )

    print(
        f"Invalid Stop             : {invalid_stop}"
    )

    assert invalid_entry == 0
    assert invalid_price == 0
    assert invalid_stop == 0


# ============================================================
# SAMPLE SIGNALS
# ============================================================

print("\n====================================")
print("🔎 SAMPLE HISTORICAL SIGNALS")
print("====================================")

if historical_signals.empty:
    print("No historical signals found.")
else:
    display(
        historical_signals[
            [
                "Symbol",
                "SignalDate",
                "EntryDate",
                "SignalClose",
                "EntryPrice",
                "Score",
                "WeeklyConfirm",
                "ATR14",
                "StopPrice",
                "Shares",
                "RiskAmount",
                "Target1",
                "Target2",
            ]
        ].tail(15)
    )


# ============================================================
# FINAL CHECK
# ============================================================

print("\n====================================")
print("✅ Historical Signal Generation Ready")
print("====================================")

📡 HISTORICAL SIGNAL GENERATION



📊 HISTORICAL SIGNAL SUMMARY
Total signals : 4,222
Symbols       : 19
Score >= 3    : 4,222
Weekly confirm: 1,614
Date range    : 2021-11-03 → 2026-10-06

🔬 LOOKAHEAD SANITY CHECK
EntryDate <= SignalDate : 0
Invalid EntryPrice      : 0
Invalid Stop             : 0

🔎 SAMPLE HISTORICAL SIGNALS


,Symbol,SignalDate,EntryDate,SignalClose,EntryPrice,Score,WeeklyConfirm,ATR14,StopPrice,Shares,RiskAmount,Target1,Target2
4207,PLANB.BK,2026-09-30,2026-10-01,6.95,6.85,4,False,0.291604,5.750000,1800,1979.999828,7.3295,7.8775
4208,AU.BK,2026-09-30,2026-10-01,4.16,4.16,3,False,0.042451,4.096323,24000,1528.251424,4.4512,4.7840
4209,FORTH.BK,2026-09-30,2026-10-01,14.60,14.40,3,False,0.647395,13.428908,2000,1942.183787,15.4080,16.5600
4210,SNNP.BK,2026-09-30,2026-10-01,6.65,6.60,3,False,0.095908,6.456138,13900,1999.678560,7.0620,7.5900
4211,TKN.BK,2026-09-30,2026-10-01,4.70,4.70,3,True,0.103949,4.440000,7600,1975.998116,5.0290,5.4050
4212,SPA.BK,2026-10-02,2026-10-05,2.98,2.98,4,False,0.067403,2.878895,19700,1991.759559,3.1886,3.4270
4213,TKN.BK,2026-10-02,2026-10-05,4.82,4.90,3,True,0.117283,4.620000,7100,1988.001490,5.2430,5.6350
4214,XO.BK,2026-10-02,2026-10-05,17.90,18.10,3,False,0.429316,17.456026,3100,1996.319592,19.3670,20.8150
4215,AU.BK,2026-10-05,2026-10-06,4.20,4.20,4,False,0.055155,4.100000,20000,1999.998093,4.4940,4.8300
4216,KAMART.BK,2026-10-05,2026-10-06,7.35,7.35,3,False,0.153111,6.650000,2800,1959.999466,7.8645,8.4525



✅ Historical Signal Generation Ready


In [ ]:

# ============================================================
# 11.2) FIX HISTORICAL SIGNALS
# PRESERVE ORIGINAL SIGNAL SET
# FIX STOP FROM SIGNAL-DATE DATA
# ============================================================

print("🔧 FIXING HISTORICAL SIGNALS")
print("=" * 90)

old_signals = historical_signals.copy()

print(f"Original signals : {len(old_signals):,}")

fixed_rows = []
excluded_invalid_stop = 0

for _, old in old_signals.iterrows():

    symbol = old["Symbol"]
    signal_date = pd.Timestamp(old["SignalDate"])
    entry_date = pd.Timestamp(old["EntryDate"])

    if symbol not in feature_data:
        continue

    df = feature_data[symbol].sort_index()

    if signal_date not in df.index:
        continue

    if entry_date not in df.index:
        continue

    # RECENT_LOW_5 is created on canonical_data in Cell 9.
    # feature_data may not carry that derived column, so rebuild it
    # here from signal-date history if it is missing.
    if "RECENT_LOW_5" not in df.columns:
        df = df.copy()
        df["RECENT_LOW_5"] = (
            df["Low"]
            .rolling(
                window=RECENT_LOW_LOOKBACK,
                min_periods=RECENT_LOW_LOOKBACK,
            )
            .min()
        )

    signal_row = df.loc[signal_date]
    entry_row = df.loc[entry_date]

    signal_close = float(signal_row["Close"])
    signal_atr = float(signal_row["ATR14"])
    recent_low_value = signal_row["RECENT_LOW_5"]

    if not np.isfinite(recent_low_value) or recent_low_value <= 0:
        continue

    recent_low = float(recent_low_value)
    entry_price = float(entry_row["Open"])

    # --------------------------------------------------------
    # FIXED STOP FROM SIGNAL DATE
    # --------------------------------------------------------

    stop_price = calculate_stop_price(
        entry_price=signal_close,
        atr=signal_atr,
        recent_low=recent_low,
    )

    if not np.isfinite(stop_price) or stop_price <= 0:
        excluded_invalid_stop += 1
        continue

    # A long trade cannot have its protective stop at or above
    # the actual next-day entry price. This can happen after a
    # gap-down between signal date and entry date. Exclude that
    # trade from the executable historical set rather than
    # manufacturing a stop or silently changing the strategy.
    if entry_price <= stop_price:
        excluded_invalid_stop += 1
        continue

    # --------------------------------------------------------
    # POSITION SIZE
    # Uses actual entry price + fixed stop
    # --------------------------------------------------------

    risk_per_share = entry_price - stop_price

    if risk_per_share <= 0:
        shares = 0
        lots = 0
        risk_amount = 0.0
        position_value = 0.0
        status = "INVALID_STOP"

    else:
        max_risk = INITIAL_CAPITAL * MAX_RISK_PCT

        risk_shares = int(max_risk / risk_per_share)
        capital_shares = int(INITIAL_CAPITAL / entry_price)

        shares = min(risk_shares, capital_shares)

        shares = (shares // LOT_SIZE) * LOT_SIZE
        lots = shares // LOT_SIZE

        risk_amount = risk_per_share * shares
        position_value = entry_price * shares

        status = "OK" if shares > 0 else "CAPITAL_LIMITED"

    # --------------------------------------------------------
    # TARGETS
    # --------------------------------------------------------

    target1 = entry_price * 1.07
    target2 = entry_price * 1.15

    fixed_rows.append({
        "Symbol": symbol,
        "SignalDate": signal_date,
        "EntryDate": entry_date,
        "SignalClose": signal_close,
        "EntryPrice": entry_price,
        "Score": int(old["Score"]),
        "WeeklyConfirm": bool(old["WeeklyConfirm"]),
        "ATR14": signal_atr,
        "RecentLow5": recent_low,
        "StopPrice": stop_price,
        "Shares": shares,
        "Lots": lots,
        "RiskAmount": risk_amount,
        "PositionValue": position_value,
        "Target1": target1,
        "Target2": target2,
        "Status": status,
        "EntryOpen": float(entry_row["Open"]),
        "EntryHigh": float(entry_row["High"]),
        "EntryLow": float(entry_row["Low"]),
        "EntryClose": float(entry_row["Close"]),
    })


# ------------------------------------------------------------
# SAFETY CHECK
# ------------------------------------------------------------

if not fixed_rows:
    raise RuntimeError(
        "No signals were rebuilt. Stop here and inspect Cell 11."
    )


historical_signals = pd.DataFrame(fixed_rows)

historical_signals = (
    historical_signals
    .sort_values(["SignalDate", "Symbol"])
    .reset_index(drop=True)
)


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print("📊 REBUILD SUMMARY")
print("====================================")

print(f"Original signals : {len(old_signals):,}")
print(f"Fixed signals    : {len(historical_signals):,}")
print(
    f"Difference       : "
    f"{len(historical_signals) - len(old_signals):+,}"
)


# ============================================================
# VALIDATION
# ============================================================

entry_before_signal = (
    historical_signals["EntryDate"]
    < historical_signals["SignalDate"]
).sum()

invalid_entry = (
    ~np.isfinite(historical_signals["EntryPrice"])
    | (historical_signals["EntryPrice"] <= 0)
).sum()

invalid_stop = (
    ~np.isfinite(historical_signals["StopPrice"])
    | (historical_signals["StopPrice"] <= 0)
).sum()

stop_ge_entry = (
    historical_signals["StopPrice"]
    >= historical_signals["EntryPrice"]
).sum()

risk_over_limit = (
    historical_signals["RiskAmount"]
    > INITIAL_CAPITAL * MAX_RISK_PCT + 1e-9
).sum()

capital_over_limit = (
    historical_signals["PositionValue"]
    > INITIAL_CAPITAL + 1e-9
).sum()


print("\n====================================")
print("🔍 VALIDATION")
print("====================================")

print(f"Entry before Signal : {entry_before_signal:,}")
print(f"Invalid EntryPrice  : {invalid_entry:,}")
print(f"Invalid Stop        : {invalid_stop:,}")
print(f"Stop >= Entry       : {stop_ge_entry:,}")
print(f"Risk > 2%           : {risk_over_limit:,}")
print(f"Capital > 100k      : {capital_over_limit:,}")


# ============================================================
# STOP CONSISTENCY
# ============================================================

check = historical_signals.head(10).copy()

check["ExpectedStop"] = check.apply(
    lambda r: calculate_stop_price(
        r["SignalClose"],
        r["ATR14"],
        r["RecentLow5"],
    ),
    axis=1,
)

check["StopMatch"] = np.isclose(
    check["StopPrice"],
    check["ExpectedStop"],
    rtol=1e-10,
    atol=1e-10,
)

print("\n====================================")
print("🔒 STOP CONSISTENCY")
print("====================================")

display(
    check[
        [
            "Symbol",
            "SignalDate",
            "SignalClose",
            "EntryPrice",
            "ATR14",
            "RecentLow5",
            "StopPrice",
            "ExpectedStop",
            "StopMatch",
        ]
    ]
)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n====================================")

assert len(historical_signals) + excluded_invalid_stop == len(old_signals), (
    "Signal accounting mismatch during rebuild."
)
assert entry_before_signal == 0, "EntryDate is before SignalDate."
assert invalid_entry == 0, "Invalid EntryPrice found."
assert invalid_stop == 0, "Invalid StopPrice found."
assert stop_ge_entry == 0, "StopPrice >= EntryPrice found."
assert risk_over_limit == 0, "Risk exceeds configured limit."
assert capital_over_limit == 0, "Position value exceeds initial capital."
assert check["StopMatch"].all(), "Stop-price consistency check failed."

print("✅ HISTORICAL SIGNALS FIXED")
print("CELL 11.2 PASSED")
print("====================================")

🔧 FIXING HISTORICAL SIGNALS
Original signals : 4,222



📊 REBUILD SUMMARY
Original signals : 4,222
Fixed signals    : 4,217
Difference       : -5

🔍 VALIDATION
Entry before Signal : 0
Invalid EntryPrice  : 0
Invalid Stop        : 0
Stop >= Entry       : 0
Risk > 2%           : 0
Capital > 100k      : 0

🔒 STOP CONSISTENCY


,Symbol,SignalDate,SignalClose,EntryPrice,ATR14,RecentLow5,StopPrice,ExpectedStop,StopMatch
0,DELTA.BK,2021-11-03,44.000000,44.599998,2.142172,40.799999,40.786742,40.786742,True
1,PLANB.BK,2021-11-03,6.157406,6.157406,0.206671,6.108538,5.847399,5.847399,True
2,AU.BK,2021-11-04,10.300000,10.300000,0.176290,10.200000,10.035565,10.035565,True
3,CBG.BK,2021-11-04,125.500000,126.000000,3.182256,121.500000,120.726616,120.726616,True
4,PLANB.BK,2021-11-04,6.499484,6.499484,0.223324,6.108538,6.108538,6.108538,True
5,CBG.BK,2021-11-05,125.000000,125.000000,3.169238,121.500000,120.246144,120.246144,True
6,DELTA.BK,2021-11-05,45.200001,45.599998,2.221567,40.799999,40.799999,40.799999,True
7,JMT.BK,2021-11-05,49.750000,49.000000,1.306057,45.915836,45.915836,45.915836,True
8,AU.BK,2021-11-08,10.300000,10.300000,0.172413,10.200000,10.041380,10.041380,True
9,JMART.BK,2021-11-08,42.750000,43.000000,1.386341,38.000000,38.000000,38.000000,True



✅ HISTORICAL SIGNALS FIXED
CELL 11.2 PASSED


In [ ]:

# ============================================================
# 12.1) SIGNAL CLUSTER ANALYSIS
# NO TRADE-ENGINE DEPENDENCY
# ============================================================

print("🔍 SIGNAL CLUSTER ANALYSIS")
print("=" * 90)

assert not historical_signals.empty, "historical_signals is empty."

analysis = historical_signals.copy()
analysis["SignalDate"] = pd.to_datetime(analysis["SignalDate"])
analysis["EntryDate"] = pd.to_datetime(analysis["EntryDate"])

analysis = analysis.sort_values(
    ["Symbol", "SignalDate"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Days since previous signal for the same symbol
# ------------------------------------------------------------

analysis["PrevSignalDate"] = (
    analysis.groupby("Symbol")["SignalDate"].shift(1)
)

analysis["DaysSincePrevSignal"] = (
    analysis["SignalDate"] - analysis["PrevSignalDate"]
).dt.days

analysis["ClusteredSignal"] = (
    analysis["DaysSincePrevSignal"].notna()
    & (analysis["DaysSincePrevSignal"] <= 3)
)

clustered = analysis[analysis["ClusteredSignal"]].copy()

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

total_signals = len(analysis)
clustered_count = len(clustered)
cluster_rate = clustered_count / total_signals * 100

print("\n====================================")
print("📊 SIGNAL CLUSTER SUMMARY")
print("====================================")
print(f"Total signals     : {total_signals:,}")
print(f"Clustered signals : {clustered_count:,}")
print(f"Cluster rate      : {cluster_rate:.2f}%")

# ------------------------------------------------------------
# Symbol-level summary
# ------------------------------------------------------------

symbol_cluster = (
    analysis.groupby("Symbol")
    .agg(
        Signals=("SignalDate", "count"),
        Clustered=("ClusteredSignal", "sum"),
    )
)

symbol_cluster["ClusterRate"] = (
    symbol_cluster["Clustered"]
    / symbol_cluster["Signals"]
    * 100
)

symbol_cluster = symbol_cluster.sort_values(
    ["Signals", "ClusterRate"],
    ascending=[False, False],
)

print("\n====================================")
print("📈 SIGNALS BY SYMBOL")
print("====================================")
display(symbol_cluster)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

same_symbol_order_ok = (
    analysis.groupby("Symbol")["SignalDate"]
    .apply(lambda s: s.is_monotonic_increasing)
    .all()
)

invalid_gap = (
    analysis["DaysSincePrevSignal"].notna()
    & (analysis["DaysSincePrevSignal"] < 0)
).sum()

assert same_symbol_order_ok, "Signal dates are not chronological by symbol."
assert invalid_gap == 0, "Negative signal-date gap found."
assert clustered_count <= total_signals

print("\n====================================")
print("🔎 RECENT CLUSTERED SIGNALS")
print("====================================")

if clustered.empty:
    print("No clustered signals found.")
else:
    display(
        clustered[
            [
                "Symbol",
                "SignalDate",
                "EntryDate",
                "Score",
                "WeeklyConfirm",
                "DaysSincePrevSignal",
            ]
        ].tail(30)
    )

print("\n====================================")
print("✅ CELL 12.1 PASSED")
print("====================================")

🔍 SIGNAL CLUSTER ANALYSIS

📊 SIGNAL CLUSTER SUMMARY
Total signals     : 4,217
Clustered signals : 2,276
Cluster rate      : 53.97%

📈 SIGNALS BY SYMBOL


,Signals,Clustered,ClusterRate
Symbol,,,
SAPPE.BK,277,167,60.288809
KAMART.BK,270,155,57.407407
AU.BK,270,151,55.925926
ICHI.BK,260,147,56.538462
SPA.BK,260,137,52.692308
PLANB.BK,248,131,52.822581
TKN.BK,243,132,54.320988
DITTO.BK,227,114,50.220264
DELTA.BK,224,113,50.446429



🔎 RECENT CLUSTERED SIGNALS


,Symbol,SignalDate,EntryDate,Score,WeeklyConfirm,DaysSincePrevSignal
4164,XO.BK,2025-06-27,2025-06-30,3,False,1.0
4167,XO.BK,2025-07-18,2025-07-21,3,True,1.0
4173,XO.BK,2025-11-28,2025-12-01,3,False,1.0
4175,XO.BK,2026-01-06,2026-01-07,3,False,1.0
4177,XO.BK,2026-01-21,2026-01-22,3,False,1.0
4179,XO.BK,2026-02-10,2026-02-11,3,False,1.0
4180,XO.BK,2026-02-11,2026-02-12,4,False,1.0
4183,XO.BK,2026-03-17,2026-03-18,3,False,1.0
4185,XO.BK,2026-03-31,2026-04-01,3,False,1.0
4186,XO.BK,2026-04-01,2026-04-02,4,False,1.0



✅ CELL 12.1 PASSED


In [ ]:
# ============================================================
# CELL 12.2 — NON-OVERLAPPING BACKTEST
# ============================================================

print("=" * 90)
print("🔬 NON-OVERLAPPING HISTORICAL BACKTEST")
print("=" * 90)

# ------------------------------------------------------------
# 1) Conservative trade simulator
# ------------------------------------------------------------

def simulate_trade_fixed_stop(signal, price_data):
    """
    Simulate one long trade with:
      - next-day entry
      - conservative gap/stop handling
      - target 1 tracking
      - target 2 exit
      - hard MAX_HOLDING_DAYS time exit
      - MAE/MFE measured only while the trade is actually open

    MAX_HOLDING_DAYS is interpreted as calendar days from EntryDate.
    On the maximum-holding-date bar, STOP/TARGET rules have priority;
    otherwise the position is closed at that day's Close.
    """

    entry_date = pd.Timestamp(signal["EntryDate"])
    entry_price = float(signal["EntryPrice"])
    stop_price = float(signal["StopPrice"])
    shares = int(signal["Shares"])

    if entry_price <= 0 or stop_price <= 0 or shares <= 0:
        return None

    target1 = entry_price * 1.07
    target2 = entry_price * 1.15

    future = price_data.loc[
        price_data.index >= entry_date
    ].copy()

    if future.empty:
        return None

    max_hold_date = entry_date + pd.Timedelta(
        days=MAX_HOLDING_DAYS
    )

    future = future.loc[
        future.index <= max_hold_date
    ].copy()

    if future.empty:
        return None

    target1_hit = False
    exit_date = pd.NaT
    exit_price = np.nan
    exit_reason = None

    for current_date, bar in future.iterrows():

        day_open = float(bar["Open"])
        day_high = float(bar["High"])
        day_low = float(bar["Low"])
        day_close = float(bar["Close"])

        # ----------------------------------------------------
        # Gap handling
        # ----------------------------------------------------
        # If market opens below stop, actual exit is OPEN.
        # This is conservative and avoids assuming execution
        # at a better price than the market provided.
        if day_open <= stop_price:
            exit_date = current_date
            exit_price = day_open
            exit_reason = "STOP_GAP"
            break

        # ----------------------------------------------------
        # Normal intraday stop
        # ----------------------------------------------------
        if day_low <= stop_price:
            exit_date = current_date
            exit_price = stop_price
            exit_reason = "STOP"
            break

        # ----------------------------------------------------
        # Target 2
        # ----------------------------------------------------
        if day_open >= target2:
            target1_hit = True
            exit_date = current_date
            exit_price = day_open
            exit_reason = "TARGET2_GAP"
            break

        if day_high >= target2:
            target1_hit = True
            exit_date = current_date
            exit_price = target2
            exit_reason = "TARGET2"
            break

        # ----------------------------------------------------
        # Target 1 — tracking only
        # ----------------------------------------------------
        if day_high >= target1:
            target1_hit = True

        # ----------------------------------------------------
        # Hard time exit
        # ----------------------------------------------------
        # STOP/TARGET checks above always have priority on this
        # date. If neither occurs, exit at the closing price.
        if current_date >= max_hold_date:
            exit_date = current_date
            exit_price = day_close
            exit_reason = "MAX_HOLD"
            break

    # --------------------------------------------------------
    # End of data
    # --------------------------------------------------------

    if pd.isna(exit_date):
        # This can only happen when the available dataset ends
        # before MAX_HOLDING_DAYS. Close at the final available bar.
        exit_date = future.index[-1]
        exit_price = float(future.iloc[-1]["Close"])
        exit_reason = "EOD"

    holding_days = (
        pd.Timestamp(exit_date) - entry_date
    ).days

    if holding_days > MAX_HOLDING_DAYS:
        raise AssertionError(
            f"MAX_HOLDING_DAYS violated: "
            f"{holding_days} > {MAX_HOLDING_DAYS}"
        )

    pnl = (exit_price - entry_price) * shares
    return_pct = (
        (exit_price - entry_price)
        / entry_price
        * 100
    )

    # IMPORTANT:
    # MAE/MFE must be calculated only over the period in which
    # the position was actually open. Using all future bars would
    # introduce look-ahead leakage into the diagnostics.
    held_future = price_data.loc[
        (price_data.index >= entry_date)
        & (price_data.index <= exit_date)
    ].copy()

    min_low = float(held_future["Low"].min())
    max_high = float(held_future["High"].max())

    mae_pct = (
        (min_low - entry_price)
        / entry_price
        * 100
    )

    mfe_pct = (
        (max_high - entry_price)
        / entry_price
        * 100
    )

    return {
        "Symbol": signal["Symbol"],
        "SignalDate": signal["SignalDate"],
        "EntryDate": entry_date,
        "EntryPrice": entry_price,
        "StopPrice": stop_price,
        "Shares": shares,
        "Target1": target1,
        "Target2": target2,
        "Target1Hit": target1_hit,
        "ExitDate": exit_date,
        "ExitPrice": exit_price,
        "ExitReason": exit_reason,
        "HoldingDays": holding_days,
        "PnL": pnl,
        "ReturnPct": return_pct,
        "MAE_Pct": mae_pct,
        "MFE_Pct": mfe_pct,
        "Score": signal["Score"],
        "WeeklyConfirm": signal["WeeklyConfirm"],
    }


# ------------------------------------------------------------
# 2) Backtest one symbol without overlapping trades
# ------------------------------------------------------------

def backtest_one_symbol_nonoverlap(symbol):

    signals = historical_signals[
        historical_signals["Symbol"] == symbol
    ].copy()

    signals = signals.sort_values(
        ["EntryDate", "SignalDate"]
    ).reset_index(drop=True)

    if symbol not in feature_data:
        return pd.DataFrame()

    price_data = feature_data[symbol].copy()

    trades = []
    previous_exit_date = pd.NaT
    skipped_overlap = 0

    for _, signal in signals.iterrows():

        entry_date = pd.Timestamp(signal["EntryDate"])

        # ----------------------------------------------------
        # Prevent overlapping trades
        # ----------------------------------------------------
        if not pd.isna(previous_exit_date):
            if entry_date <= previous_exit_date:
                skipped_overlap += 1
                continue

        trade = simulate_trade_fixed_stop(
            signal,
            price_data
        )

        if trade is None:
            continue

        trades.append(trade)
        previous_exit_date = trade["ExitDate"]

    result = pd.DataFrame(trades)

    return result


# ------------------------------------------------------------
# 3) Test AU.BK first
# ------------------------------------------------------------

TEST_SYMBOL = "AU.BK"

au_trades = backtest_one_symbol_nonoverlap(TEST_SYMBOL)

print("\n====================================")
print(f"📊 {TEST_SYMBOL} BACKTEST")
print("====================================")

print(
    f"Signals available : "
    f"{len(historical_signals[historical_signals['Symbol'] == TEST_SYMBOL]):,}"
)

print(f"Trades executed   : {len(au_trades):,}")

if not au_trades.empty:

    print(
        f"Total P&L         : "
        f"{au_trades['PnL'].sum():,.2f} THB"
    )

    print(
        f"Average Return    : "
        f"{au_trades['ReturnPct'].mean():.2f}%"
    )

    print(
        f"Median Return     : "
        f"{au_trades['ReturnPct'].median():.2f}%"
    )

    print(
        f"Win Rate          : "
        f"{(au_trades['ReturnPct'] > 0).mean() * 100:.2f}%"
    )

    print("\n====================================")
    print("🔎 TRADE SAMPLE")
    print("====================================")

    display(
        au_trades[
            [
                "Symbol",
                "SignalDate",
                "EntryDate",
                "EntryPrice",
                "StopPrice",
                "ExitDate",
                "ExitPrice",
                "ExitReason",
                "HoldingDays",
                "Shares",
                "PnL",
                "ReturnPct",
                "Target1Hit",
            ]
        ].head(15)
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    invalid_exit = (
        au_trades["ExitDate"] < au_trades["EntryDate"]
    ).sum()

    invalid_return = (
        ~pd.to_numeric(
            au_trades["ReturnPct"],
            errors="coerce"
        ).notna()
    ).sum()

    invalid_holding = (
        au_trades["HoldingDays"] > MAX_HOLDING_DAYS
    ).sum()

    print("\n====================================")
    print("🛡️ BACKTEST VALIDATION")
    print("====================================")
    print(f"Exit before Entry : {invalid_exit}")
    print(f"Invalid Return    : {invalid_return}")
    print(f"Over Max Holding  : {invalid_holding}")

    assert invalid_exit == 0
    assert invalid_return == 0
    assert invalid_holding == 0

    print("\n====================================")
    print("✅ CELL 12.2 PASSED")
    print("====================================")
    print(
        "Non-overlapping backtest is valid for the test symbol."
    )

else:
    print(
        "\n⚠️ No executable trades found for",
        TEST_SYMBOL
    )



🔬 NON-OVERLAPPING HISTORICAL BACKTEST



📊 AU.BK BACKTEST
Signals available : 270
Trades executed   : 70
Total P&L         : -16,541.96 THB
Average Return    : -0.58%
Median Return     : -1.74%
Win Rate          : 34.29%

🔎 TRADE SAMPLE


,Symbol,SignalDate,EntryDate,EntryPrice,StopPrice,ExitDate,ExitPrice,ExitReason,HoldingDays,Shares,PnL,ReturnPct,Target1Hit
0,AU.BK,2021-11-04,2021-11-05,10.30,10.035565,2021-11-25,10.400000,MAX_HOLD,20,7500,749.995708,0.970868,False
1,AU.BK,2021-12-13,2021-12-14,10.10,9.758271,2021-12-30,9.950000,EOD,16,5800,-870.003319,-1.485154,False
2,AU.BK,2022-01-04,2022-01-05,9.90,9.638555,2022-01-25,10.100000,MAX_HOLD,20,7600,1520.005798,2.020210,True
3,AU.BK,2022-01-26,2022-01-27,10.40,9.950000,2022-02-15,10.200000,EOD,19,4400,-879.999161,-1.923075,False
4,AU.BK,2022-02-17,2022-02-18,10.20,9.860083,2022-03-07,9.860083,STOP,17,5800,-1971.519645,-3.332521,False
5,AU.BK,2022-03-07,2022-03-08,9.75,9.499579,2022-03-28,10.100000,MAX_HOLD,20,7900,2765.003014,3.589748,False
6,AU.BK,2022-03-31,2022-04-01,10.30,9.892327,2022-04-21,10.600000,MAX_HOLD,20,4900,1470.000935,2.912623,False
7,AU.BK,2022-04-21,2022-04-22,10.50,9.950000,2022-05-06,9.950000,STOP,14,3600,-1980.000687,-5.238097,False
8,AU.BK,2022-06-01,2022-06-02,10.10,9.832265,2022-06-14,9.832265,STOP,12,7400,-1981.239484,-2.650842,False
9,AU.BK,2022-06-28,2022-06-29,9.70,9.400000,2022-07-05,9.400000,STOP,6,6600,-1980.001259,-3.092786,False



🛡️ BACKTEST VALIDATION
Exit before Entry : 0
Invalid Return    : 0
Over Max Holding  : 0

✅ CELL 12.2 PASSED
Non-overlapping backtest is valid for the test symbol.


In [ ]:

# ============================================================
# CELL 12.3 — TRADE ENGINE DIAGNOSTICS
# ============================================================

print("=" * 90)
print("🔎 TRADE ENGINE DIAGNOSTICS — AU.BK")
print("=" * 90)

if au_trades.empty:
    print("⚠️ No trades available.")
else:

    df = au_trades.copy()

    # --------------------------------------------------------
    # 1) Exit distribution
    # --------------------------------------------------------

    exit_counts = (
        df["ExitReason"]
        .value_counts()
        .rename_axis("ExitReason")
        .reset_index(name="Trades")
    )

    exit_counts["Pct"] = (
        exit_counts["Trades"] / len(df) * 100
    )

    print("\n====================================")
    print("📊 EXIT DISTRIBUTION")
    print("====================================")

    display(exit_counts)

    # --------------------------------------------------------
    # 2) Return distribution
    # --------------------------------------------------------

    print("\n====================================")
    print("📈 RETURN DISTRIBUTION")
    print("====================================")

    print(f"Best trade        : {df['ReturnPct'].max():.2f}%")
    print(f"Worst trade       : {df['ReturnPct'].min():.2f}%")
    print(f"Average           : {df['ReturnPct'].mean():.2f}%")
    print(f"Median            : {df['ReturnPct'].median():.2f}%")
    print(f"Win rate          : {(df['ReturnPct'] > 0).mean() * 100:.2f}%")
    print(f"Loss rate         : {(df['ReturnPct'] < 0).mean() * 100:.2f}%")
    print(f"Flat              : {(df['ReturnPct'] == 0).mean() * 100:.2f}%")

    # --------------------------------------------------------
    # 3) Risk characteristics
    # --------------------------------------------------------

    df["RiskPct"] = (
        (df["EntryPrice"] - df["StopPrice"])
        / df["EntryPrice"]
        * 100
    )

    print("\n====================================")
    print("🛡️ STOP / RISK DISTRIBUTION")
    print("====================================")

    print(f"Average Stop Risk : {df['RiskPct'].mean():.2f}%")
    print(f"Median Stop Risk  : {df['RiskPct'].median():.2f}%")
    print(f"Minimum Stop Risk : {df['RiskPct'].min():.2f}%")
    print(f"Maximum Stop Risk : {df['RiskPct'].max():.2f}%")

    # --------------------------------------------------------
    # 4) Target1 reached but eventually lost
    # --------------------------------------------------------

    target1_then_loss = df[
        (df["Target1Hit"] == True) &
        (df["ReturnPct"] < 0)
    ].copy()

    print("\n====================================")
    print("⚠️ TARGET1 HIT → EVENTUAL LOSS")
    print("====================================")

    print(
        f"Count: {len(target1_then_loss):,} "
        f"({len(target1_then_loss) / len(df) * 100:.2f}%)"
    )

    if not target1_then_loss.empty:
        display(
            target1_then_loss[
                [
                    "SignalDate",
                    "EntryDate",
                    "EntryPrice",
                    "StopPrice",
                    "ExitDate",
                    "ExitPrice",
                    "ExitReason",
                    "ReturnPct",
                    "PnL",
                ]
            ].sort_values("ReturnPct").head(10)
        )

    # --------------------------------------------------------
    # 5) Large stop-distance trades
    # --------------------------------------------------------

    print("\n====================================")
    print("⚠️ LARGEST STOP DISTANCES")
    print("====================================")

    display(
        df[
            [
                "SignalDate",
                "EntryDate",
                "EntryPrice",
                "StopPrice",
                "RiskPct",
                "ExitReason",
                "ReturnPct",
                "PnL",
            ]
        ]
        .sort_values("RiskPct", ascending=False)
        .head(10)
    )

    # --------------------------------------------------------
    # 6) Trade duration
    # --------------------------------------------------------

    df["HoldingDays"] = (
        pd.to_datetime(df["ExitDate"])
        - pd.to_datetime(df["EntryDate"])
    ).dt.days

    print("\n====================================")
    print("⏱️ HOLDING PERIOD")
    print("====================================")

    print(f"Average days     : {df['HoldingDays'].mean():.1f}")
    print(f"Median days      : {df['HoldingDays'].median():.1f}")
    print(f"Maximum days     : {df['HoldingDays'].max():.0f}")
    print(f"MAX_HOLDING_DAYS : {MAX_HOLDING_DAYS}")

    over_limit = int(
        (df["HoldingDays"] > MAX_HOLDING_DAYS).sum()
    )

    print(f"Over max holding : {over_limit}")

    # --------------------------------------------------------
    # 7) Basic assertions
    # --------------------------------------------------------

    assert (df["EntryDate"] <= df["ExitDate"]).all()

    assert (df["HoldingDays"] >= 0).all()
    assert (df["HoldingDays"] <= MAX_HOLDING_DAYS).all()
    assert over_limit == 0
    assert df["RiskPct"].notna().all()
    assert df["ReturnPct"].notna().all()

    print("\n====================================")
    print("✅ CELL 12.3 PASSED")
    print("====================================")
    print("Trade-engine diagnostics completed.")


🔎 TRADE ENGINE DIAGNOSTICS — AU.BK

📊 EXIT DISTRIBUTION


,ExitReason,Trades,Pct
0,STOP,28,40.000000
1,MAX_HOLD,25,35.714286
2,EOD,11,15.714286
3,TARGET2,3,4.285714
4,STOP_GAP,3,4.285714



📈 RETURN DISTRIBUTION
Best trade        : 15.00%
Worst trade       : -11.81%
Average           : -0.58%
Median            : -1.74%
Win rate          : 34.29%
Loss rate         : 65.71%
Flat              : 0.00%

🛡️ STOP / RISK DISTRIBUTION
Average Stop Risk : 4.77%
Median Stop Risk  : 4.14%
Minimum Stop Risk : 1.11%
Maximum Stop Risk : 16.57%

⚠️ TARGET1 HIT → EVENTUAL LOSS
Count: 0 (0.00%)

⚠️ LARGEST STOP DISTANCES


,SignalDate,EntryDate,EntryPrice,StopPrice,RiskPct,ExitReason,ReturnPct,PnL
41,2024-08-13,2024-08-14,8.75,7.30,16.571426,MAX_HOLD,4.000004,455.000496
52,2025-08-06,2025-08-07,7.20,6.35,11.805555,STOP,-11.805555,-1954.999781
49,2025-04-16,2025-04-17,9.85,8.80,10.659900,MAX_HOLD,-7.614213,-1425.000000
51,2025-07-03,2025-07-04,6.65,5.95,10.526320,MAX_HOLD,2.255640,420.000267
58,2026-01-08,2026-01-09,5.05,4.60,8.910896,MAX_HOLD,-2.970299,-660.000420
40,2024-07-31,2024-08-01,8.15,7.50,7.975456,STOP,-7.975456,-1949.998856
39,2024-06-25,2024-06-26,7.25,6.70,7.586210,TARGET2,15.000000,3915.000000
13,2022-09-15,2022-09-16,10.90,10.10,7.339443,STOP,-7.339443,-1999.998093
15,2022-10-27,2022-10-28,11.30,10.50,7.079648,STOP,-7.079648,-1920.000458
14,2022-10-06,2022-10-07,10.60,9.85,7.075471,MAX_HOLD,6.603772,1819.999504



⏱️ HOLDING PERIOD
Average days     : 14.1
Median days      : 16.0
Maximum days     : 20
MAX_HOLDING_DAYS : 20
Over max holding : 0

✅ CELL 12.3 PASSED
Trade-engine diagnostics completed.


In [ ]:
# ============================================================
# CELL 12.4 — ALL-SYMBOL NON-OVERLAPPING BACKTEST
# Research validation across the complete watchlist
# ============================================================

print("=" * 90)
print("📊 ALL-SYMBOL NON-OVERLAPPING BACKTEST")
print("=" * 90)

all_trades = []
symbol_summary = []

for symbol in WATCHLIST:
    trades = backtest_one_symbol_nonoverlap(symbol)

    signal_count = int(
        (historical_signals["Symbol"] == symbol).sum()
    )

    trade_count = len(trades)

    if not trades.empty:
        all_trades.append(trades)

        symbol_summary.append({
            "Symbol": symbol,
            "Signals": signal_count,
            "Trades": trade_count,
            "PnL": float(trades["PnL"].sum()),
            "AvgReturnPct": float(trades["ReturnPct"].mean()),
            "WinRatePct": float((trades["ReturnPct"] > 0).mean() * 100),
            "MaxHoldingDays": int(trades["HoldingDays"].max()),
        })
    else:
        symbol_summary.append({
            "Symbol": symbol,
            "Signals": signal_count,
            "Trades": 0,
            "PnL": 0.0,
            "AvgReturnPct": np.nan,
            "WinRatePct": np.nan,
            "MaxHoldingDays": 0,
        })

if all_trades:
    all_trades_df = (
        pd.concat(all_trades, ignore_index=True)
        .sort_values(["EntryDate", "Symbol"])
        .reset_index(drop=True)
    )
else:
    all_trades_df = pd.DataFrame()

symbol_summary_df = pd.DataFrame(symbol_summary)

print("\n====================================")
print("📈 PORTFOLIO-LEVEL SUMMARY")
print("====================================")

print(f"Symbols tested     : {len(WATCHLIST)}")
print(f"Total trades       : {len(all_trades_df):,}")

if not all_trades_df.empty:
    print(f"Total P&L          : {all_trades_df['PnL'].sum():,.2f} THB")
    print(f"Average return     : {all_trades_df['ReturnPct'].mean():.2f}%")
    print(f"Median return      : {all_trades_df['ReturnPct'].median():.2f}%")
    print(f"Win rate           : {(all_trades_df['ReturnPct'] > 0).mean() * 100:.2f}%")
    print(f"Max holding days   : {all_trades_df['HoldingDays'].max():.0f}")

    over_limit = int(
        (all_trades_df["HoldingDays"] > MAX_HOLDING_DAYS).sum()
    )

    invalid_dates = int(
        (all_trades_df["ExitDate"] < all_trades_df["EntryDate"]).sum()
    )

    invalid_returns = int(
        (~pd.to_numeric(
            all_trades_df["ReturnPct"],
            errors="coerce"
        ).notna()).sum()
    )

    print(f"Over max holding   : {over_limit}")
    print(f"Invalid exit dates : {invalid_dates}")
    print(f"Invalid returns    : {invalid_returns}")

    print("\n====================================")
    print("🚪 EXIT REASONS")
    print("====================================")
    display(
        all_trades_df["ExitReason"]
        .value_counts()
        .rename_axis("ExitReason")
        .reset_index(name="Trades")
    )

    print("\n====================================")
    print("🏆 BY SYMBOL")
    print("====================================")
    display(
        symbol_summary_df
        .sort_values(["PnL", "WinRatePct"], ascending=[False, False])
        .reset_index(drop=True)
    )

    assert over_limit == 0
    assert invalid_dates == 0
    assert invalid_returns == 0
    assert all_trades_df["HoldingDays"].ge(0).all()
    assert all_trades_df["HoldingDays"].le(MAX_HOLDING_DAYS).all()

    print("\n====================================")
    print("✅ CELL 12.4 PASSED")
    print("====================================")
    print("All-symbol non-overlapping backtest passed validation.")
else:
    print("⚠️ No executable trades found across the watchlist.")


📊 ALL-SYMBOL NON-OVERLAPPING BACKTEST



📈 PORTFOLIO-LEVEL SUMMARY
Symbols tested     : 19
Total trades       : 1,267
Total P&L          : -93,967.01 THB
Average return     : -0.15%
Median return      : -2.65%
Win rate           : 37.41%
Max holding days   : 20
Over max holding   : 0
Invalid exit dates : 0
Invalid returns    : 0

🚪 EXIT REASONS


,ExitReason,Trades
0,STOP,525
1,MAX_HOLD,396
2,TARGET2,167
3,EOD,126
4,STOP_GAP,41
5,TARGET2_GAP,12



🏆 BY SYMBOL


,Symbol,Signals,Trades,PnL,AvgReturnPct,WinRatePct,MaxHoldingDays
0,SAPPE.BK,277,68,41769.163414,1.736932,47.058824,20
1,DELTA.BK,224,71,28253.190284,1.589693,46.478873,20
2,TKN.BK,243,69,25260.970384,0.115515,34.782609,20
3,XO.BK,213,71,20328.070085,0.749531,39.436620,20
4,ICHI.BK,260,68,11146.705055,0.524193,48.529412,20
5,PLANB.BK,248,72,10244.484320,0.542150,41.666667,20
6,CBG.BK,194,63,-1502.563570,-0.124961,34.920635,20
7,DITTO.BK,227,75,-1816.222563,0.194454,41.333333,20
8,KLINIQ.BK,170,50,-3054.612516,0.029152,38.000000,20
9,KAMART.BK,270,76,-3938.822355,0.027835,39.473684,20



✅ CELL 12.4 PASSED
All-symbol non-overlapping backtest passed validation.


## CHECKPOINT — 12.2 PASSED

Cells 12.1 and 12.2 have been validated.

Run **Cell 12.3 only** next.

If Cell 12.3 prints `CELL 12.3 PASSED`, the next stage can be added.

In [ ]:
# ============================================================
# CELL 12.5 — WALK-FORWARD STRATEGY OPTIMIZATION
# ============================================================

print("=" * 90)
print("🧪 WALK-FORWARD STRATEGY OPTIMIZATION")
print("=" * 90)

# Fast research grid. Parameters are selected on TRAIN only,
# filtered on VALIDATION, then evaluated once on unseen TEST.
OPT_GRID = {
    "min_score": [3, 4, 5],
    "weekly_confirm": [False, True],
    "atr_mult": [1.2, 1.5],
    "target2_pct": [0.10, 0.15],
    "max_hold_days": [15, 20],
}
OPT_MIN_TRADES = 25

# ------------------------------------------------------------
# Chronological split
# ------------------------------------------------------------
dates = pd.to_datetime(
    historical_signals["SignalDate"]
).sort_values().unique()

assert len(dates) >= 100, "Not enough historical signal dates."

n = len(dates)
train_end = dates[int(n * 0.60) - 1]
valid_end = dates[int(n * 0.80) - 1]
test_start = dates[int(n * 0.80)]

print(f"Train end      : {pd.Timestamp(train_end).date()}")
print(f"Validation end : {pd.Timestamp(valid_end).date()}")
print(f"Test start     : {pd.Timestamp(test_start).date()}")

# ------------------------------------------------------------
# Cache signal sets. This avoids recalculating indicators and
# signal rows for every target/holding combination.
# ------------------------------------------------------------
signal_cache = {}

for min_score in OPT_GRID["min_score"]:
    for weekly_confirm in OPT_GRID["weekly_confirm"]:
        for atr_mult in OPT_GRID["atr_mult"]:

            key = (min_score, weekly_confirm, atr_mult)
            rows = []

            for symbol, df in canonical_data.items():

                mask = df["CANONICAL_SCORE"].ge(min_score)

                if weekly_confirm:
                    mask &= df[
                        "CANONICAL_WEEKLY_CONFIRM"
                    ].fillna(False)

                signal_positions = np.flatnonzero(
                    mask.to_numpy()
                )

                for pos in signal_positions:

                    if pos + 1 >= len(df):
                        continue

                    sig = df.iloc[pos]
                    ent = df.iloc[pos + 1]

                    entry = float(ent["Open"])

                    stop = calculate_stop_price(
                        float(sig["Close"]),
                        float(sig["ATR14"]),
                        float(sig["RECENT_LOW_5"]),
                        atr_mult,
                    )

                    if (
                        not np.isfinite(entry)
                        or entry <= 0
                        or not np.isfinite(stop)
                        or stop <= 0
                        or stop >= entry
                    ):
                        continue

                    position = calculate_position_size(
                        INITIAL_CAPITAL,
                        entry,
                        stop,
                    )

                    if position["shares"] <= 0:
                        continue

                    rows.append({
                        "Symbol": symbol,
                        "SignalDate": df.index[pos],
                        "EntryDate": df.index[pos + 1],
                        "EntryPrice": entry,
                        "StopPrice": float(stop),
                        "Shares": int(position["shares"]),
                    })

            signal_cache[key] = (
                pd.DataFrame(rows)
                .sort_values(
                    ["EntryDate", "SignalDate", "Symbol"]
                )
                .reset_index(drop=True)
                if rows
                else pd.DataFrame()
            )

print(
    f"Cached signal sets: {len(signal_cache)}"
)

# ------------------------------------------------------------
# Fast trade simulation
# ------------------------------------------------------------
def _opt_run(
    signals,
    target2_pct,
    max_hold_days,
    start_date,
    end_date,
):
    if signals.empty:
        return {
            "PnL": 0.0,
            "Trades": 0,
            "WinRatePct": np.nan,
        }

    selected = signals[
        (signals["SignalDate"] >= start_date)
        & (signals["SignalDate"] <= end_date)
    ].sort_values(
        ["EntryDate", "SignalDate", "Symbol"]
    )

    pnl = 0.0
    trades = 0
    wins = 0
    previous_exit = {}

    for _, sig in selected.iterrows():

        symbol = sig["Symbol"]
        entry_date = pd.Timestamp(sig["EntryDate"])

        if (
            symbol in previous_exit
            and entry_date <= previous_exit[symbol]
        ):
            continue

        df = canonical_data[symbol]

        end_date_hold = (
            entry_date
            + pd.Timedelta(days=max_hold_days)
        )

        future = df.loc[
            (df.index >= entry_date)
            & (df.index <= end_date_hold)
        ]

        if future.empty:
            continue

        entry = float(sig["EntryPrice"])
        stop = float(sig["StopPrice"])
        shares = int(sig["Shares"])
        target2 = entry * (1 + target2_pct)

        exit_date = pd.Timestamp(future.index[-1])
        exit_price = float(future.iloc[-1]["Close"])

        for current_date, bar in future.iterrows():

            op = float(bar["Open"])
            hi = float(bar["High"])
            lo = float(bar["Low"])
            cl = float(bar["Close"])

            if op <= stop:
                exit_date = pd.Timestamp(current_date)
                exit_price = op
                break

            if lo <= stop:
                exit_date = pd.Timestamp(current_date)
                exit_price = stop
                break

            if op >= target2:
                exit_date = pd.Timestamp(current_date)
                exit_price = op
                break

            if hi >= target2:
                exit_date = pd.Timestamp(current_date)
                exit_price = target2
                break

            if current_date >= end_date_hold:
                exit_date = pd.Timestamp(current_date)
                exit_price = cl
                break

        trade_pnl = (
            exit_price - entry
        ) * shares

        pnl += trade_pnl
        trades += 1
        wins += int(trade_pnl > 0)

        previous_exit[symbol] = exit_date

    return {
        "PnL": float(pnl),
        "Trades": int(trades),
        "WinRatePct": (
            float(wins / trades * 100)
            if trades
            else np.nan
        ),
    }

# ------------------------------------------------------------
# Evaluate grid
# ------------------------------------------------------------
import itertools

variants = [
    dict(zip(OPT_GRID.keys(), values))
    for values in itertools.product(
        *OPT_GRID.values()
    )
]

train_rows = []
valid_rows = []

for v in variants:

    key = (
        v["min_score"],
        v["weekly_confirm"],
        v["atr_mult"],
    )

    train_result = _opt_run(
        signal_cache[key],
        v["target2_pct"],
        v["max_hold_days"],
        dates[0],
        train_end,
    )

    valid_result = _opt_run(
        signal_cache[key],
        v["target2_pct"],
        v["max_hold_days"],
        train_end + np.timedelta64(1, "D"),
        valid_end,
    )

    train_rows.append({
        **v,
        **{
            f"Train_{k}": value
            for k, value in train_result.items()
        },
    })

    valid_rows.append({
        **v,
        **{
            f"Valid_{k}": value
            for k, value in valid_result.items()
        },
    })

train_df = pd.DataFrame(train_rows)
valid_df = pd.DataFrame(valid_rows)

results = train_df.merge(
    valid_df,
    on=list(OPT_GRID.keys()),
)

eligible = results[
    (results["Train_Trades"] >= OPT_MIN_TRADES)
    & (results["Valid_Trades"] >= 10)
    & (results["Valid_PnL"] > 0)
].copy()

if eligible.empty:
    print(
        "⚠️ No candidate passed the positive-validation filter."
    )

    eligible = results[
        (results["Train_Trades"] >= OPT_MIN_TRADES)
        & (results["Valid_Trades"] >= 10)
    ].copy()

assert not eligible.empty, (
    "No viable strategy candidate found."
)

best = eligible.sort_values(
    ["Train_PnL", "Valid_PnL"],
    ascending=[False, False],
).iloc[0]

BEST_STRATEGY = {
    "min_score": int(best["min_score"]),
    "weekly_confirm": bool(best["weekly_confirm"]),
    "atr_mult": float(best["atr_mult"]),
    "target2_pct": float(best["target2_pct"]),
    "max_hold_days": int(best["max_hold_days"]),
}

print("\n====================================")
print("🏆 SELECTED STRATEGY")
print("====================================")
print(BEST_STRATEGY)
print(
    f"Train P&L      : "
    f"{best['Train_PnL']:,.2f} THB"
)
print(
    f"Validation P&L : "
    f"{best['Valid_PnL']:,.2f} THB"
)

# ------------------------------------------------------------
# FINAL UNSEEN TEST — run exactly once
# ------------------------------------------------------------
best_key = (
    BEST_STRATEGY["min_score"],
    BEST_STRATEGY["weekly_confirm"],
    BEST_STRATEGY["atr_mult"],
)

optimized_test = _opt_run(
    signal_cache[best_key],
    BEST_STRATEGY["target2_pct"],
    BEST_STRATEGY["max_hold_days"],
    test_start,
    dates[-1],
)

baseline = _opt_run(
    signal_cache[(3, False, 1.5)],
    0.15,
    20,
    test_start,
    dates[-1],
)

print("\n====================================")
print("🧪 FINAL UNSEEN TEST")
print("====================================")
print(
    f"Baseline P&L   : "
    f"{baseline['PnL']:,.2f} THB"
)
print(
    f"Optimized P&L  : "
    f"{optimized_test['PnL']:,.2f} THB"
)
print(
    f"Delta P&L      : "
    f"{optimized_test['PnL'] - baseline['PnL']:,.2f} THB"
)
print(
    f"Optimized trades: "
    f"{optimized_test['Trades']}"
)
print(
    f"Optimized win rate: "
    f"{optimized_test['WinRatePct']:.2f}%"
)

if optimized_test["PnL"] > baseline["PnL"]:
    print(
        "\n✅ OPTIMIZED STRATEGY BEATS BASELINE "
        "ON UNSEEN TEST DATA"
    )
else:
    print(
        "\n⚠️ BASELINE REMAINS BETTER "
        "ON UNSEEN TEST DATA"
    )

print("\nTop validation candidates:")
display(
    eligible.sort_values(
        ["Valid_PnL", "Train_PnL"],
        ascending=[False, False],
    )[
        list(OPT_GRID.keys())
        + ["Train_PnL", "Valid_PnL",
           "Train_Trades", "Valid_Trades"]
    ].head(10)
)


🧪 WALK-FORWARD STRATEGY OPTIMIZATION
Train end      : 2024-10-04
Validation end : 2025-10-27
Test start     : 2025-10-28


Cached signal sets: 12


⚠️ No candidate passed the positive-validation filter.

🏆 SELECTED STRATEGY
{'min_score': 4, 'weekly_confirm': False, 'atr_mult': 1.2, 'target2_pct': 0.15, 'max_hold_days': 20}
Train P&L      : 85,297.22 THB
Validation P&L : -38,200.77 THB



🧪 FINAL UNSEEN TEST
Baseline P&L   : -67,395.13 THB
Optimized P&L  : -170.21 THB
Delta P&L      : 67,224.91 THB
Optimized trades: 130
Optimized win rate: 36.92%

✅ OPTIMIZED STRATEGY BEATS BASELINE ON UNSEEN TEST DATA

Top validation candidates:


,min_score,weekly_confirm,atr_mult,target2_pct,max_hold_days,Train_PnL,Valid_PnL,Train_Trades,Valid_Trades
15,3,True,1.5,0.15,20,45995.590130,-1383.312390,303,63
11,3,True,1.2,0.15,20,36501.166549,-2497.044803,310,64
45,5,True,1.5,0.10,20,23387.884381,-11667.313141,56,11
41,5,True,1.2,0.10,20,15186.292303,-11742.005556,57,11
47,5,True,1.5,0.15,20,17416.722677,-12327.313561,52,11
43,5,True,1.2,0.15,20,9457.247613,-12402.005976,54,11
20,4,False,1.5,0.10,15,59324.390892,-12838.566897,368,80
40,5,True,1.2,0.10,15,13482.163885,-13022.007387,57,11
42,5,True,1.2,0.15,15,9477.496836,-13022.007387,54,11
44,5,True,1.5,0.10,15,23222.272529,-13067.314858,56,11


In [ ]:
# ============================================================
# CELL 12.6 — SYMBOL FILTER OPTIMIZATION
# ============================================================

print("=" * 90)
print("🎯 SYMBOL FILTER OPTIMIZATION")
print("=" * 90)

# Use the already-selected strategy from Cell 12.5.
assert "BEST_STRATEGY" in globals()

base_key = (
    BEST_STRATEGY["min_score"],
    BEST_STRATEGY["weekly_confirm"],
    BEST_STRATEGY["atr_mult"],
)

base_signals = signal_cache[base_key]

def _symbol_pnl(
    signals,
    target2_pct,
    max_hold_days,
    start_date,
    end_date,
):
    rows = []

    if signals.empty:
        return pd.DataFrame()

    for symbol in sorted(signals["Symbol"].unique()):

        result = _opt_run(
            signals[signals["Symbol"] == symbol],
            target2_pct,
            max_hold_days,
            start_date,
            end_date,
        )

        rows.append({
            "Symbol": symbol,
            **result,
        })

    return pd.DataFrame(rows)

train_symbol = _symbol_pnl(
    base_signals,
    BEST_STRATEGY["target2_pct"],
    BEST_STRATEGY["max_hold_days"],
    dates[0],
    train_end,
)

valid_symbol = _symbol_pnl(
    base_signals,
    BEST_STRATEGY["target2_pct"],
    BEST_STRATEGY["max_hold_days"],
    train_end + np.timedelta64(1, "D"),
    valid_end,
)

symbol_eval = train_symbol.merge(
    valid_symbol,
    on="Symbol",
    suffixes=("_Train", "_Valid"),
)

# Rank symbols by TRAIN P&L only.
symbol_eval = symbol_eval.sort_values(
    "PnL_Train",
    ascending=False,
).reset_index(drop=True)

print("\n====================================")
print("📊 SYMBOL TRAIN / VALIDATION")
print("====================================")
display(symbol_eval)

# Test several portfolio widths.
K_GRID = [5, 8, 10, 12, 15, 19]

filter_results = []

for k in K_GRID:

    symbols = symbol_eval.head(k)["Symbol"].tolist()

    train_pnl = symbol_eval.head(k)["PnL_Train"].sum()
    valid_pnl = symbol_eval.head(k)["PnL_Valid"].sum()

    filter_results.append({
        "K": k,
        "Symbols": symbols,
        "TrainPnL": float(train_pnl),
        "ValidPnL": float(valid_pnl),
        "ValidTrades": int(
            symbol_eval.head(k)["Trades_Valid"].sum()
        ),
    })

filter_df = pd.DataFrame(filter_results)

print("\n====================================")
print("🏆 SYMBOL FILTER CANDIDATES")
print("====================================")
display(
    filter_df[
        ["K", "TrainPnL", "ValidPnL", "ValidTrades"]
    ].sort_values(
        "ValidPnL",
        ascending=False,
    )
)

eligible_filter = filter_df[
    (filter_df["TrainPnL"] > 0)
    & (filter_df["ValidTrades"] >= 20)
].copy()

if eligible_filter.empty:
    eligible_filter = filter_df[
        filter_df["ValidTrades"] >= 20
    ].copy()

assert not eligible_filter.empty

selected_filter = eligible_filter.sort_values(
    ["ValidPnL", "TrainPnL"],
    ascending=[False, False],
).iloc[0]

SELECTED_SYMBOLS = list(
    selected_filter["Symbols"]
)

print("\n====================================")
print("🎯 SELECTED SYMBOL UNIVERSE")
print("====================================")
print(
    f"K = {int(selected_filter['K'])}"
)
print(
    f"Symbols = {SELECTED_SYMBOLS}"
)
print(
    f"Validation P&L = "
    f"{selected_filter['ValidPnL']:,.2f} THB"
)

# ------------------------------------------------------------
# Final unseen test
# ------------------------------------------------------------
test_signals = base_signals[
    base_signals["Symbol"].isin(SELECTED_SYMBOLS)
]

filtered_test = _opt_run(
    test_signals,
    BEST_STRATEGY["target2_pct"],
    BEST_STRATEGY["max_hold_days"],
    test_start,
    dates[-1],
)

unfiltered_test = _opt_run(
    base_signals,
    BEST_STRATEGY["target2_pct"],
    BEST_STRATEGY["max_hold_days"],
    test_start,
    dates[-1],
)

print("\n====================================")
print("🧪 FINAL TEST — SYMBOL FILTER")
print("====================================")
print(
    f"Unfiltered P&L : "
    f"{unfiltered_test['PnL']:,.2f} THB"
)
print(
    f"Filtered P&L   : "
    f"{filtered_test['PnL']:,.2f} THB"
)
print(
    f"Delta P&L      : "
    f"{filtered_test['PnL'] - unfiltered_test['PnL']:,.2f} THB"
)
print(
    f"Filtered trades: "
    f"{filtered_test['Trades']}"
)
print(
    f"Filtered win rate: "
    f"{filtered_test['WinRatePct']:.2f}%"
)

if filtered_test["PnL"] > unfiltered_test["PnL"]:
    print(
        "\n✅ SYMBOL FILTER IMPROVES UNSEEN TEST P&L"
    )
else:
    print(
        "\n⚠️ SYMBOL FILTER DOES NOT IMPROVE UNSEEN TEST P&L"
    )


🎯 SYMBOL FILTER OPTIMIZATION



📊 SYMBOL TRAIN / VALIDATION


,Symbol,PnL_Train,Trades_Train,WinRatePct_Train,PnL_Valid,Trades_Valid,WinRatePct_Valid
0,TKN.BK,24271.393995,21,42.857143,-1039.997721,5,40.000000
1,SAPPE.BK,23173.791256,26,53.846154,8350.000000,5,80.000000
2,SPA.BK,9156.714038,24,45.833333,-1998.739666,1,0.000000
3,XO.BK,7897.116824,21,38.095238,3959.999943,2,50.000000
4,FORTH.BK,7004.508500,16,56.250000,1440.368680,4,50.000000
5,DITTO.BK,6796.459236,17,35.294118,-4789.999924,5,20.000000
6,ICHI.BK,6788.490038,23,60.869565,79.995346,4,50.000000
7,MASTER.BK,5144.721756,9,55.555556,-3954.929796,2,0.000000
8,MEB.BK,5054.040469,8,50.000000,-7296.672321,4,0.000000
9,MOSHI.BK,4994.886398,10,60.000000,-11692.266294,8,0.000000



🏆 SYMBOL FILTER CANDIDATES


,K,TrainPnL,ValidPnL,ValidTrades
0,5,71503.524612,10711.631236,17
1,8,90233.195642,2046.696862,28
2,10,100282.122509,-16942.241753,40
4,15,100913.016903,-24640.740260,58
3,12,103051.239044,-25917.244424,48
5,19,85297.217537,-38200.770179,79



🎯 SELECTED SYMBOL UNIVERSE
K = 8
Symbols = ['TKN.BK', 'SAPPE.BK', 'SPA.BK', 'XO.BK', 'FORTH.BK', 'DITTO.BK', 'ICHI.BK', 'MASTER.BK']
Validation P&L = 2,046.70 THB



🧪 FINAL TEST — SYMBOL FILTER
Unfiltered P&L : -170.21 THB
Filtered P&L   : 25,242.06 THB
Delta P&L      : 25,412.28 THB
Filtered trades: 55
Filtered win rate: 45.45%

✅ SYMBOL FILTER IMPROVES UNSEEN TEST P&L


## PRODUCTION STRATEGY — OPTIMIZED

The current production configuration is frozen from the walk-forward research:

- Minimum score: **4**
- Weekly confirmation: **OFF**
- ATR stop multiplier: **1.2×**
- Target 2: **15%**
- Maximum holding: **20 calendar days**
- Production universe: **8 symbols**
  - TKN.BK
  - SAPPE.BK
  - SPA.BK
  - XO.BK
  - FORTH.BK
  - DITTO.BK
  - ICHI.BK
  - MASTER.BK

Unseen test result for this configuration + selected symbol universe:
**+25,242.06 THB**, versus **-170.21 THB** without the symbol filter and **-68,350.12 THB** for the original baseline on the same unseen period.

This configuration should remain frozen until a new out-of-sample research cycle is completed.

In [ ]:
# ============================================================
# CELL 12.7 — PRODUCTION ROBUSTNESS / RISK DIAGNOSTICS
# ============================================================

print("=" * 90)
print("🛡️ PRODUCTION ROBUSTNESS & RISK DIAGNOSTICS")
print("=" * 90)

assert "BEST_STRATEGY" in globals()
assert "SELECTED_SYMBOLS" in globals()

prod_signals = signal_cache[
    (
        BEST_STRATEGY["min_score"],
        BEST_STRATEGY["weekly_confirm"],
        BEST_STRATEGY["atr_mult"],
    )
]
prod_signals = prod_signals[
    prod_signals["Symbol"].isin(SELECTED_SYMBOLS)
].copy()

def _trade_records(signals, start_date, end_date):
    rows=[]
    previous_exit={}
    selected=signals[
        (signals["SignalDate"]>=start_date)&
        (signals["SignalDate"]<=end_date)
    ].sort_values(["EntryDate","SignalDate","Symbol"])

    for _,sig in selected.iterrows():
        symbol=sig["Symbol"]
        entry_date=pd.Timestamp(sig["EntryDate"])
        if symbol in previous_exit and entry_date<=previous_exit[symbol]:
            continue
        df=canonical_data[symbol]
        end_hold=entry_date+pd.Timedelta(days=BEST_STRATEGY["max_hold_days"])
        future=df.loc[(df.index>=entry_date)&(df.index<=end_hold)]
        if future.empty: continue
        entry=float(sig["EntryPrice"]); stop=float(sig["StopPrice"])
        shares=int(sig["Shares"]); target2=entry*(1+BEST_STRATEGY["target2_pct"])
        exit_date=pd.Timestamp(future.index[-1]); exit_price=float(future.iloc[-1]["Close"]); reason="EOD"
        for dt,bar in future.iterrows():
            op,hi,lo,cl=map(float,[bar["Open"],bar["High"],bar["Low"],bar["Close"]])
            if op<=stop: exit_date=pd.Timestamp(dt); exit_price=op; reason="STOP_GAP"; break
            if lo<=stop: exit_date=pd.Timestamp(dt); exit_price=stop; reason="STOP"; break
            if op>=target2: exit_date=pd.Timestamp(dt); exit_price=op; reason="TARGET2_GAP"; break
            if hi>=target2: exit_date=pd.Timestamp(dt); exit_price=target2; reason="TARGET2"; break
            if dt>=end_hold: exit_date=pd.Timestamp(dt); exit_price=cl; reason="MAX_HOLD"; break
        pnl=(exit_price-entry)*shares
        rows.append({
            "Symbol":symbol,"SignalDate":pd.Timestamp(sig["SignalDate"]),
            "EntryDate":entry_date,"ExitDate":exit_date,
            "EntryPrice":entry,"ExitPrice":exit_price,"StopPrice":stop,
            "Shares":shares,"PnL":pnl,
            "ReturnPct":(exit_price/entry-1)*100,
            "HoldDays":(exit_date-entry_date).days,
            "Reason":reason,
            "RiskPct":max(0,(entry-stop)/entry*100),
        })
        previous_exit[symbol]=exit_date
    return pd.DataFrame(rows)

prod_trades=_trade_records(prod_signals,dates[0],dates[-1])
assert not prod_trades.empty

# Sequential equity using initial capital. This is a trade-level risk diagnostic,
# not yet the final multi-symbol capital allocator.
equity=INITIAL_CAPITAL
curve=[]
peak=equity
max_dd=0.0
for _,t in prod_trades.sort_values("EntryDate").iterrows():
    equity += float(t.PnL)
    peak=max(peak,equity)
    dd=(equity-peak)/peak*100
    max_dd=min(max_dd,dd)
    curve.append((t.EntryDate,equity,dd))
equity_curve=pd.DataFrame(curve,columns=["Date","Equity","DrawdownPct"])

gross_profit=prod_trades.loc[prod_trades.PnL>0,"PnL"].sum()
gross_loss=-prod_trades.loc[prod_trades.PnL<0,"PnL"].sum()
profit_factor=(gross_profit/gross_loss) if gross_loss>0 else np.inf
expectancy=prod_trades.PnL.mean()
max_loss=prod_trades.PnL.min()
max_gain=prod_trades.PnL.max()
avg_win=prod_trades.loc[prod_trades.PnL>0,"PnL"].mean()
avg_loss=prod_trades.loc[prod_trades.PnL<0,"PnL"].mean()

print(f"Trades              : {len(prod_trades):,}")
print(f"Net P&L              : {prod_trades.PnL.sum():,.2f} THB")
print(f"Win rate             : {(prod_trades.PnL>0).mean()*100:.2f}%")
print(f"Profit factor        : {profit_factor:.3f}")
print(f"Expectancy / trade   : {expectancy:,.2f} THB")
print(f"Average winner       : {avg_win:,.2f} THB")
print(f"Average loser        : {avg_loss:,.2f} THB")
print(f"Best trade           : {max_gain:,.2f} THB")
print(f"Worst trade          : {max_loss:,.2f} THB")
print(f"Max drawdown         : {max_dd:.2f}%")
print(f"Ending equity        : {equity:,.2f} THB")
print(f"Average holding      : {prod_trades.HoldDays.mean():.2f} days")
print(f"Median holding       : {prod_trades.HoldDays.median():.2f} days")
print(f"Max holding          : {prod_trades.HoldDays.max():.0f} days")
print(f"Average planned stop : {prod_trades.RiskPct.mean():.2f}%")
print(f"Max planned stop     : {prod_trades.RiskPct.max():.2f}%")

print("\nExit reasons:")
print(prod_trades.Reason.value_counts().to_string())

# Yearly stability.
prod_trades["Year"]=prod_trades.EntryDate.dt.year
yearly=prod_trades.groupby("Year").agg(
    Trades=("PnL","size"),
    PnL=("PnL","sum"),
    WinRatePct=("PnL",lambda x:(x>0).mean()*100),
)
print("\nYearly stability:")
print(yearly.to_string())

# Stress test: degrade every exit by 0.5% and 1.0% to model slippage/fees.
for friction in [0.005,0.010]:
    stressed=(prod_trades.ExitPrice*(1-friction)-prod_trades.EntryPrice)*prod_trades.Shares
    print(f"Stress {friction*100:.1f}% exit friction P&L: {stressed.sum():,.2f} THB")

print("\n🛡️ ROBUSTNESS DIAGNOSTICS COMPLETE")


🛡️ PRODUCTION ROBUSTNESS & RISK DIAGNOSTICS


Trades              : 240
Net P&L              : 117,521.96 THB
Win rate             : 47.08%
Profit factor        : 1.630
Expectancy / trade   : 489.67 THB
Average winner       : 2,690.89 THB
Average loser        : -1,541.72 THB
Best trade           : 22,770.00 THB
Worst trade          : -2,160.00 THB
Max drawdown         : -11.83%
Ending equity        : 217,521.96 THB
Average holding      : 13.84 days
Median holding       : 18.00 days
Max holding          : 20 days
Average planned stop : 7.54%
Max planned stop     : 34.51%

Exit reasons:
Reason
MAX_HOLD       101
STOP            80
TARGET2         35
EOD             19
STOP_GAP         3
TARGET2_GAP      2

Yearly stability:
      Trades           PnL  WinRatePct
Year                                  
2021      11  11150.422535   54.545455
2022      56  37677.428346   46.428571
2023      53  37065.953333   54.716981
2024      41  -2710.610288   36.585366
2025      30  12250.704845   50.000000
2026      49  22088.056739   44.897959
St